# MIZAN - Google Colab: RAG Build + QLoRA Fine-tuning
### Tunisian Legal AI | Full pipeline on free T4 GPU

**What this notebook does:**
1. Mounts Google Drive and reads your `.txt` files from `MyDrive/mizan/maraja/`
2. Parses, chunks, and embeds them -> saves FAISS index to Drive
3. Auto-generates a fine-tuning dataset from the index + HF Inference API
4. Fine-tunes `Qwen2.5-3B-Instruct` with QLoRA (fits T4 15GB)
5. Merges LoRA weights and pushes the final model to HuggingFace Hub
6. Downloads ready-to-use `rag_engine.py` + `mizan_data.zip` for your laptop

---

**Before running:**
- Runtime -> Change runtime type -> **T4 GPU** (or better)
- Upload your `.txt` files to Google Drive at: `My Drive/mizan/maraja/`
- Have your HuggingFace token ready (needs **write** access)

**To retrain after adding new files:**
Re-run Section A (rebuild index) then Section B (new dataset) then Section C (retrain).


## Step 0 - Verify GPU
> If this fails, go to Runtime -> Change runtime type -> T4 GPU

In [3]:
import subprocess, sys, torch

r = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                   capture_output=True, text=True)
print("GPU:", r.stdout.strip())
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram:.1f} GB")
    assert vram >= 12, f"Need >= 12GB VRAM for QLoRA. Got {vram:.1f}GB. Change runtime to T4."
else:
    sys.exit("ERROR: No GPU. Go to Runtime -> Change runtime type -> T4 GPU")


GPU: Tesla T4, 15360 MiB
CUDA available: True
VRAM: 15.6 GB


## Step 1 - Mount Google Drive

In [4]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=False)

DRIVE_MIZAN  = Path("/content/drive/MyDrive/LegalMind")
MARAJA_DIR   = DRIVE_MIZAN / "maraja"
DATA_DIR     = DRIVE_MIZAN / "data"
LORA_DIR     = DRIVE_MIZAN / "mizan-lora"
LOCAL_DIR    = Path("/content/mizan")

for d in [MARAJA_DIR, DATA_DIR, LORA_DIR, LOCAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

txt_files = sorted(MARAJA_DIR.glob("*.txt"))
print(f"Drive mounted.")
print(f"  maraja/  : {MARAJA_DIR}")
print(f"  data/    : {DATA_DIR}")
print(f"  Found {len(txt_files)} .txt files:")
for f in txt_files:
    print(f"    {f.name}  ({f.stat().st_size/1024:.0f} KB)")

if not txt_files:
    print()
    print("NO FILES FOUND. Upload your .txt files to:")
    print(f"  Google Drive -> mizan -> maraja -> [your files]")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.
  maraja/  : /content/drive/MyDrive/LegalMind/maraja
  data/    : /content/drive/MyDrive/LegalMind/data
  Found 58 .txt files:
    Acte_deces.txt  (3 KB)
    Aide_judiciaire.txt  (5 KB)
    ArbitrageArabe.txt  (102 KB)
    Attesta_non_faillite.txt  (5 KB)
    COCArabe.txt  (0 KB)
    CommerceArabe.txt  (423 KB)
    Communaute_biens.txt  (11 KB)
    Condamne_prison_amende_20102009.txt  (7 KB)
    Conseil_prud_homme.txt  (15 KB)
    DIPArabe.txt  (32 KB)
    EnfantArabe.txt  (227 KB)
    Extinction_peine_2012.txt  (4 KB)
    Grace_habilitation.txt  (7 KB)
    Immat_fonciere_20102009.txt  (9 KB)
    Injonction_a_payer_2012.txt  (7 KB)
    Litige_tribu_competent.txt  (15 KB)
    NationaliteArabe.txt  (36 KB)
    Nom_pat_enf_abondonnes_2012.txt  (10 KB)
    Ordonnance_sur_requete.txt  (4 KB)
    Peine_de_reparation_penale_20102009.txt  (11 KB)
    P

## Step 2 - Install dependencies
> ~2-3 min. Run once per Colab session.

In [5]:
%pip install -q sentence-transformers faiss-cpu huggingface_hub langdetect tqdm numpy datasets
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q --no-deps trl peft accelerate bitsandbytes xformers
print("All packages installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 59.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 90.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 138.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 132.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 118.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

## Step 3 - HuggingFace token & username

To securely store your HuggingFace token, add it to Colab's secrets manager. Click on the '🔑' icon in the left panel, then select 'Add a new secret'. Set the name to `HF_TOKEN` and paste your HuggingFace token (ensure it has **write** access) as the value.

In [6]:
import os, json, re, hashlib, pickle
from huggingface_hub import login, InferenceClient
from google.colab import userdata

# Securely fetch the token from Colab Secrets
HF_TOKEN = userdata.get('HF_TOKEN')

HF_USERNAME = userdata.get('HF_USERNAME')
MODEL_REPO  = f"{HF_USERNAME}/legalmind-chatbot"

# Login using the secret token
login(token=HF_TOKEN, add_to_git_credential=False)
os.environ["HF_TOKEN"] = HF_TOKEN

print(f"Logged in successfully. Model will be pushed to -> {MODEL_REPO}")

Logged in successfully. Model will be pushed to -> rayenrayenrayenrayen/legalmind-chatbot


In [7]:
# Original cell 'c8' will be replaced by the updated one. This is a placeholder for the agent to know that the original 'c8' content should be effectively removed or modified.
# The actual modification will be handled by the generate_cells command inserting the corrected code after 'm7' and effectively overriding 'c8'.

In [8]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

# Ensure the token is not None or empty
if not HF_TOKEN:
    raise ValueError("HuggingFace token not found. Please set 'HF_TOKEN' in Colab secrets.")

# Existing code from cell c8
import getpass, os, json, re, hashlib, pickle
from huggingface_hub import login, InferenceClient

HF_USERNAME = userdata.get('HF_USERNAME')
MODEL_REPO  = f"{HF_USERNAME}/legalmind-chatbot"

login(token=HF_TOKEN, add_to_git_credential=False)
os.environ["HF_TOKEN"] = HF_TOKEN

print(f"Logged in. Model will be pushed to -> {MODEL_REPO}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in. Model will be pushed to -> rayenrayenrayenrayen/legalmind-chatbot


---
## SECTION A - Build FAISS Index from txt files
Re-run this section alone whenever you add new source files.


In [9]:
from pathlib import Path
from langdetect import detect, LangDetectException
from tqdm import tqdm
import re, json, hashlib, pickle, numpy as np, faiss
from sentence_transformers import SentenceTransformer

CHUNK_SIZE      = 300
CHUNK_OVERLAP   = 60
MIN_CHUNK_CHARS = 80
EMBED_MODEL     = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

DOCS_PATH   = DATA_DIR / "documents.json"
CHUNKS_PATH = DATA_DIR / "chunks.json"
INDEX_PATH  = DATA_DIR / "legal_index.faiss"
META_PATH   = DATA_DIR / "legal_metadata.pkl"

ARTICLE_SPLIT = re.compile(
    r'(?=^\s*(?:'
    r'\u0627\u0644\u0641\u0635\u0644\s+\d+'   # الفصل
    r'|\u0627\u0644\u0645\u0627\u062f\u0629\s+\d+'   # المادة
    r'|\u0627\u0644\u0628\u0627\u0628\s+\w+'   # الباب
    r'|[Aa]rticle\s+\d+'
    r'|[Aa]rt\.?\s+\d+'
    r'|[Cc]hapitre\s+[IVX\d]+'
    r'|[Tt]itre\s+[IVX\d]+'
    r'|[Ss]ection\s+[IVX\d]+'
    r'))',
    re.MULTILINE
)

LABEL_RE = re.compile(
    r'^\s*(\u0627\u0644\u0641\u0635\u0644\s+\d+'
    r'|\u0627\u0644\u0645\u0627\u062f\u0629\s+\d+'
    r'|\u0627\u0644\u0628\u0627\u0628\s+\w+'
    r'|[Aa]rticle\s+\d+'
    r'|[Aa]rt\.?\s+\d+'
    r'|[Cc]hapitre\s+[IVX\d]+'
    r'|[Tt]itre\s+[IVX\d]+'
    r'|[Ss]ection\s+[IVX\d]+)',
    re.MULTILINE
)

def clean(t):
    t = re.sub(r'[ \t]+', ' ', t)
    t = re.sub(r'\n{3,}', '\n\n', t)
    t = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', t)
    return t.strip()

def lang_of(text):
    ar = len(re.findall(r'[\u0600-\u06FF]', text)) / max(len(text), 1)
    if ar > 0.20: return "ar"
    try: return detect(text)
    except LangDetectException: return "fr"

def split_articles(text):
    parts = ARTICLE_SPLIT.split(text)
    result = []
    for p in parts:
        p = p.strip()
        if not p: continue
        m = LABEL_RE.match(p)
        result.append((m.group(1).strip() if m else "", p))
    return result if len(result) > 1 else [("", text)]

def infer_title(fname):
    s = re.sub(r'[-_]', ' ', Path(fname).stem)
    return re.sub(r'\s+', ' ', s).strip().title()

txt_files = sorted(MARAJA_DIR.glob("*.txt"))
print(f"Processing {len(txt_files)} files...")
documents = []

for fpath in tqdm(txt_files, desc="Parsing"):
    try:
        text = fpath.read_text(encoding="utf-8", errors="ignore")
    except Exception as e:
        print(f"  SKIP {fpath.name}: {e}"); continue
    text = clean(text)
    if len(text) < 150: continue
    title = infer_title(fpath.name)
    lang  = lang_of(text[:500])
    for label, section in split_articles(text):
        section = clean(section)
        if len(section) < 80: continue
        uid = hashlib.md5(f"{fpath.name}|{label}|{section[:30]}".encode()).hexdigest()[:12]
        documents.append({
            "id": uid, "title": title, "source": fpath.name,
            "language": lang, "article": label, "text": section, "url": "",
        })

with open(DOCS_PATH, "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

fr = sum(1 for d in documents if d["language"]=="fr")
ar = sum(1 for d in documents if d["language"]=="ar")
print(f"Sections: {len(documents)} | French: {fr} | Arabic: {ar}")


Processing 58 files...


Parsing: 100%|██████████| 58/58 [00:50<00:00,  1.15it/s]


Sections: 616 | French: 0 | Arabic: 616


In [10]:
# Smart chunking: keep short articles whole, split long ones
all_chunks, kept_whole, was_split = [], 0, 0
for doc in tqdm(documents, desc="Chunking"):
    words = doc["text"].split()
    if len(words) <= CHUNK_SIZE:
        c = {**doc, "chunk_id": f"{doc['id']}_c0", "doc_id": doc["id"], "chunk_index": 0}
        all_chunks.append(c); kept_whole += 1
    else:
        step = max(CHUNK_SIZE - CHUNK_OVERLAP, 1)
        for i, start in enumerate(range(0, len(words), step)):
            w = words[start:start+CHUNK_SIZE]
            if not w: break
            t = " ".join(w).strip()
            if len(t) < MIN_CHUNK_CHARS: continue
            c = {**doc, "text": t,
                 "chunk_id": f"{doc['id']}_c{i}",
                 "doc_id": doc["id"], "chunk_index": i}
            all_chunks.append(c)
        was_split += 1

avg = sum(len(c["text"]) for c in all_chunks) / max(len(all_chunks), 1)
print(f"Chunks: {len(all_chunks)} (whole: {kept_whole}, split: {was_split}) | avg {avg:.0f} chars")
with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)


Chunking: 100%|██████████| 616/616 [00:00<00:00, 7299.59it/s]

Chunks: 2942 (whole: 237, split: 379) | avg 1465 chars


In [11]:
print(f"Loading {EMBED_MODEL}...")
embedder = SentenceTransformer(EMBED_MODEL)
DIM = embedder.get_sentence_embedding_dimension()
print(f"  dim={DIM}")

texts = [c["text"] for c in all_chunks]
print(f"Encoding {len(texts)} chunks on GPU...")

BATCH = 256
emb_list = []
for i in tqdm(range(0, len(texts), BATCH), desc="Embedding"):
    e = embedder.encode(texts[i:i+BATCH],
                        normalize_embeddings=True,
                        show_progress_bar=False,
                        convert_to_numpy=True, device="cuda")
    emb_list.append(e)

embeddings = np.vstack(emb_list).astype("float32")
index = faiss.IndexFlatIP(DIM)
index.add(embeddings)

faiss.write_index(index, str(INDEX_PATH))
metadata = {"embed_model":EMBED_MODEL,"embed_dim":DIM,
            "n_docs":len(documents),"n_chunks":len(all_chunks),"chunks":all_chunks}
with open(META_PATH,"wb") as f: pickle.dump(metadata, f)

print(f"FAISS index: {index.ntotal} vectors -> {INDEX_PATH}")
print(f"Metadata                             -> {META_PATH}")


Loading sentence-transformers/paraphrase-multilingual-mpnet-base-v2...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_13284/1637163697.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DIM = embedder.get_sentence_embedding_dimension()


  dim=768
Encoding 2942 chunks on GPU...


Embedding: 100%|██████████| 12/12 [00:25<00:00,  2.11s/it]


FAISS index: 2942 vectors -> /content/drive/MyDrive/LegalMind/data/legal_index.faiss
Metadata                             -> /content/drive/MyDrive/LegalMind/data/legal_metadata.pkl


### A1 - Quick retrieval sanity check (no LLM)

In [12]:
def retrieve_raw(query, k=4):
    q = embedder.encode([query], normalize_embeddings=True,
                        convert_to_numpy=True, device="cuda").astype("float32")
    scores, idxs = index.search(q, k)
    return [(float(scores[0][i]), all_chunks[idxs[0][i]]) for i in range(k) if idxs[0][i] != -1]

for q in ["prescription action civile", "contrat bail", "code penal vol",
          "\u0627\u0633\u062a\u0626\u0646\u0627\u0641 \u0645\u062f\u0646\u064a"]:
    print(f"Q: {q}")
    for sc, c in retrieve_raw(q, k=2):
        art = f" | {c['article']}" if c.get("article") else ""
        print(f"  [{sc:.3f}] {c['source']}{art}")
        print(f"           {c['text'][:100]}...")
    print()

print("If all scores are below 0.25, check your txt file encoding (open in Notepad, save as UTF-8).")


Q: prescription action civile
  [0.685] manuel_proced__officier_poli_jud.txt | الفصل 87
           دورية بجميع المؤسسات المشار اليها والسهر علسى مراعساة مقتضسيات التشريع الخساص بالميحدان الصيدلي وتحر...
  [0.626] manuel_proced_enregis_deci.txt
           تعيين مقدارها من طرف رئيس المحكمة وبدون لزوم لإجراءات جديدة . كماجاء بالفصل 357 من مجلة الحقوق العين...

Q: contrat bail
  [0.664] CommerceArabe.txt | الفصل 459
           المجلة. الفصل 467 تحدد المحكمة أجلا كراس شروط من @أ[٥ لتحرير قبل القضائي . ويجب أن تضمن به شروط الكر...
  [0.649] societeArabe.txt | الفصل 56
           نفقة الطالب الفصل ٠66- لا يمكن إطلاع العموم على معنى أحكام , هذا الباب 1 بالنسبة لإجراءات التسو يقال...

Q: code penal vol
  [0.649] PenalArabe.txt | الفصل 19
           بهذا 103 --- Page 104 --- القانون أو الجرائم المرتبطة بها بإبلاغ السلط ذات النظر بإرشادات أو معلومات...
  [0.642] PenalArabe.txt | الفصل 248
           أو إمضاء أو إحدى الأوراق المبينة 0@ 283 من هذه المجلة . الفصل ٠285- يمكن تطبيق العقوبات المبينة با

---
## SECTION B - Generate Fine-tuning Dataset
Calls Mistral-7B via HF Inference API to produce gold Q&A pairs.
Takes ~15-30 min depending on how many seed questions there are.
Re-run after adding new source files to expand the dataset.


In [13]:
client = InferenceClient(token=HF_TOKEN)
# Switching to a model that might be available on a different provider to bypass credit limit
LLM_BASE = "mistralai/Mistral-7B-Instruct-v0.2"
DATASET_PATH = DATA_DIR / "finetune_dataset.jsonl"

SEED_QUESTIONS = [
    # Re-running primarily the missing Arabic and skipped French questions
    "Comment contester une decision administrative en Tunisie ?",
    "Quels sont les delais de recours devant le tribunal administratif ?",
    "Qu'est-ce que le recours for exces de pouvoir en Tunisie ?",
    "Quelles sont les conditions du mariage en Tunisie ?",
    "Comment se deroule une procedure de divorce en Tunisie ?",
    "Comment est fixee la pension alimentaire en Tunisie ?",
    "Quelles sont les regles de la tutelle en droit tunisien ?",
    "Quelles sont les regles de succession en Tunisie ?",
    "Qui sont les heritiers reservataires en droit tunisien ?",
    "Quelles sont les conditions du licenciement en Tunisie ?",
    "Quelle est la duree legale du travail en Tunisie ?",
    "Quels sont les droits du salarie lors d'un licenciement abusif ?",
    "\u0645\u0627 \u0647\u064a \u0625\u062c\u0631\u0627\u0621\u0627\u062a \u0627\u0644\u0627\u0633\u062a\u0626\u0646\u0627\u0641 \u0641\u064a \u0627\u0644\u0642\u0636\u0627\u064a\u0627 \u0627\u0644\u0645\u062f\u0646\u064a\u0629 \u0627\u0644\u062a\u0648\u0646\u0633\u064a\u0629 \u061f",
    "\u0645\u0627 \u0647\u064a \u062d\u0642\u0648\u0642 \u0627\u0644\u0645\u062a\u0647\u0645 \u0641\u064a \u0645\u0631\u062d\u0644\u0629 \u0627\u0644\u0627\u0633\u062a\u062c\u0648\u0627\u0628 \u061f",
    "\u0645\u0627 \u0647\u064a \u0634\u0631\u0648\u0637 \u0635\u062d\u0629 \u0639\u0642\u062f \u0627\u0644\u0628\u064a\u0639 \u0641\u064a \u0627\u0644\u0642\u0627\u0646\u0648\u0646 \u0627\u0644\u062a\u0648\u0646\u0633\u064a \u061f",
    "\u0645\u0627 \u0647\u064a \u0627\u0644\u0639\u0642\u0648\u0628\u0627\u062a \u0627\u0644\u0645\u0642\u0631\u0631\u0629 \u0644\u0644\u0633\u0631\u0642\u0629 \u0641\u064a \u0627\u0644\u0642\u0627\u0646\u0648\u0646 \u0627\u0644\u062a\u0648\u0646\u0633\u064a \u061f",
    "\u0643\u064a\u0641 \u064a\u062a\u0645 \u062a\u0623\u0633\u064a\u0633 \u0634\u0631\u0643\u0629 \u0630\u0627\u062a \u0645\u0633\u0624\u0648\u0644\u064a\u0629 \u0645\u062d\u062f\u0648\u062f\u0629 \u0641\u064a \u062a\u0648\u0646\u0633 \u061f",
    "\u0645\u0627 \u0647\u064a \u0645\u0648\u0627\u0639\u064a\u062f \u0627\u0644\u062a\u0642\u0627\u062f\u0645 \u0641\u064a \u0627\u0644\u0642\u0627\u0646\u0648\u0646 \u0627\u0644\u062a\u0648\u0646\u0633\u064a \u061f",
    "\u0645\u0627 \u0647\u064a \u0625\u062c\u0631\u0627\u0621\u0627\u062a \u0627\u0644\u0637\u0644\u0627\u0642 \u0641\u064a \u0627\u0644\u0642\u0627\u0646\u0648\u0646 \u0627\u0644\u062a\u0648\u0646\u0633\u064a \u061f",
    "\u0645\u0627 \u0647\u064a \u062d\u0642\u0648\u0642 \u0627\u0644\u0639\u0627\u0645\u0644 \u0639\u0642\u062f \u0627\u0644\u0611\u0635\u0644 \u0627\u0644\u062a\u0639\u0633\u0641\u064a \u061f",
]

def gen_answer(question, contexts):
    lang = lang_of(question)
    sys_msg = (
        "\u0623\u0646\u062a MIZAN \u0645\u0633\u0627\u0639\u062f \u0642\u0627\u0646\u0648\u0646\u064a \u062a\u0648\u0646\u0633\u064a. \u0623\u062c\u0628 \u0628\u0646\u0627\u0621 \u0639\u0644\u0649 \u0627\u0644\u0646\u0635\u0648\u0635 \u0641\u0642\u0637. \u0627\u0630\u0643\u0631 \u0627\u0633\u0645 \u0627\u0644\u0646\u0635 \u0648\u0627\u0644\u0641\u0635\u0644."
        if lang == "ar" else
        "Tu es MIZAN, assistant juridique tunisien. Reponds UNIQUEMENT sur la base des textes. Cite la source et l'article."
    )
    ctx = "".join(f"[{i}] {c['source']}" + (f" | {c['article']}" if c.get("article") else "") + f"\n{c['text']}\n\n" for i,c in enumerate(contexts,1))
    try:
        r = client.chat_completion(
            model=LLM_BASE, messages=[{"role":"system","content":sys_msg}, {"role":"user","content":f"Textes:\n\n{ctx}\nQuestion: {question}"}],
            max_tokens=400, temperature=0.1
        )
        return r.choices[0].message.content.strip()
    except Exception as e:
        return None

# Load existing dataset to append
existing_data = []
if DATASET_PATH.exists():
    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        existing_data = [json.loads(line) for line in f]

print(f"Current dataset size: {len(existing_data)}. Attempting to fill missing questions...")

new_count = 0
for q in tqdm(SEED_QUESTIONS):
    if any(d['instruction'] == q for d in existing_data): continue

    ctxs = [all_chunks[idxs] for idxs in index.search(
        embedder.encode([q], normalize_embeddings=True, convert_to_numpy=True, device="cuda").astype("float32"), 4
    )[1][0] if idxs != -1]

    gold = gen_answer(q, ctxs)
    if gold:
        item = {"instruction":q, "context":"\n\n".join(f"[{c['source']}] {c.get('article','')}\n{c['text']}" for c in ctxs), "output":gold, "language":lang_of(q)}
        existing_data.append(item)
        with open(DATASET_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
        new_count += 1

print(f"\nFinished. New total: {len(existing_data)} (Added {new_count})")

Current dataset size: 9. Attempting to fill missing questions...


100%|██████████| 20/20 [00:03<00:00,  5.63it/s]


Finished. New total: 9 (Added 0)


---
## SECTION C - QLoRA Fine-tuning
Model: Qwen2.5-3B-Instruct | Expected time: 25-50 min on T4


In [14]:
from unsloth import FastLanguageModel
import torch
MAX_SEQ_LEN = 2048

print("Loading base model with 4-bit quantization...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)
print(f"Base model loaded ({sum(p.numel() for p in model.parameters())/1e6:.0f}M params)")


/tmp/ipykernel_13284/3482725431.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading base model with 4-bit quantization...
==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Base model loaded (1800M params)


In [15]:
from unsloth import FastLanguageModel
import torch
MAX_SEQ_LEN = 2048

print("Loading base model with 4-bit quantization...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)
print(f"Base model loaded ({sum(p.numel() for p in model.parameters())/1e6:.0f}M params)")


Loading base model with 4-bit quantization...
==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Base model loaded (1800M params)


In [16]:
from unsloth import FastLanguageModel
import torch
MAX_SEQ_LEN = 4096

print("Loading base model with 4-bit quantization...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)
print(f"Base model loaded ({sum(p.numel() for p in model.parameters())/1e6:.0f}M params)")


Loading base model with 4-bit quantization...
==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Base model loaded (1800M params)


In [17]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"LoRA: {trainable/1e6:.1f}M trainable / {total/1e6:.0f}M total ({100*trainable/total:.2f}%)")


Unsloth 2026.4.6 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


LoRA: 29.9M trainable / 1830M total (1.64%)


In [18]:
from datasets import Dataset
import math

with open(DATASET_PATH, encoding="utf-8") as f:
    raw = [json.loads(l) for l in f]

def fmt(ex):
    lang = ex.get("language","fr")
    sys_msg = (
        "\u0623\u0646\u062a MIZAN \u0645\u0633\u0627\u0639\u062f \u0642\u0627\u0646\u0648\u0646\u064a \u062a\u0648\u0646\u0633\u064a. \u0623\u062c\u0628 \u0628\u0646\u0627\u0621 \u0639\u0644\u0649 \u0627\u0644\u0646\u0635\u0648\u0635. \u0627\u0630\u0643\u0631 \u0627\u0644\u0645\u0635\u062f\u0631 \u0648\u0627\u0644\u0641\u0635\u0644."
        if lang == "ar" else
        "Tu es MIZAN, assistant juridique expert en droit tunisien. Reponds uniquement sur la base des textes fournis. Cite la source et l'article."
    )
    user = f"Textes juridiques pertinents:\n\n{ex['context']}\n\nQuestion: {ex['instruction']}"
    return {"text": (
        f"<|im_start|>system\n{sys_msg}<|im_end|>\n"
        f"<|im_start|>user\n{user}<|im_end|>\n"
        f"<|im_start|>assistant\n{ex['output']}<|im_end|>"
    )}

hf_ds = Dataset.from_list(raw).map(fmt)
n = len(hf_ds)
num_epochs = 5 if n < 100 else 4 if n < 250 else 3
total_steps = math.ceil(n / (2 * 4)) * num_epochs
print(f"Dataset: {n} | Epochs: {num_epochs} | Steps: {total_steps} | Est: ~{total_steps*8//60} min")


Map:   0%|          | 0/9 [00:00<?, ? examples/s]

Dataset: 9 | Epochs: 5 | Steps: 10 | Est: ~1 min


In [19]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = hf_ds,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LEN,
    dataset_num_proc   = 2,
    packing            = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio                = 0.05,
        num_train_epochs            = num_epochs,
        learning_rate               = 2e-4,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 5,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "cosine",
        seed                        = 42,
        output_dir                  = str(LOCAL_DIR / "checkpoints"),
        save_strategy               = "epoch",
        report_to                   = "none",
    ),
)
print("Trainer ready. Starting training...")
stats = trainer.train()
print(f"Done! Final loss: {stats.training_loss:.4f} | Time: {stats.metrics['train_runtime']:.0f}s")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/9 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Trainer ready. Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9 | Num Epochs = 5 | Total steps = 10
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,3.144486
10,2.690470


Done! Final loss: 2.9175 | Time: 68s


In [20]:
# Quick inference test with fine-tuned model
FastLanguageModel.for_inference(model)

def local_answer(q, ctxs):
    lang = lang_of(q)
    sys_msg = (
        "\u0623\u0646\u062a MIZAN \u0645\u0633\u0627\u0639\u062f \u0642\u0627\u0646\u0648\u0646\u064a \u062a\u0648\u0646\u0633\u064a."
        if lang == "ar" else
        "Tu es MIZAN, assistant juridique tunisien. Cite la source et l'article."
    )
    ctx = "\n\n".join(f"[{c['source']}]" + (f" {c['article']}" if c.get("article") else "") + f"\n{c['text']}" for c in ctxs)
    prompt = (
        f"<|im_start|>system\n{sys_msg}<|im_end|>\n"
        f"<|im_start|>user\nTextes:\n\n{ctx}\n\nQuestion: {q}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=300, temperature=0.1,
                     do_sample=False, repetition_penalty=1.3,
                     pad_token_id=tokenizer.eos_token_id)
    raw_out = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    # Remove leaked metadata prefix like "AR14:11" or "الفصل 14" at the start
    raw_out = re.sub(r'^[\w\d\u0600-\u06FF]+\d+[\:\.\d]*\s*\n', '', raw_out).strip()
    return raw_out

TEST_Q = [
    "Quels sont les delais de prescription en matiere civile ?",
    "Comment contester une decision administrative ?",
    "\u0645\u0627 \u0647\u064a \u062d\u0642\u0648\u0642 \u0627\u0644\u0645\u062a\u0647\u0645 \u0641\u064a \u0645\u0631\u062d\u0644\u0629 \u0627\u0644\u0627\u0633\u062a\u062c\u0648\u0627\u0628 \u061f",
]
for q in TEST_Q:
    q_vec = embedder.encode([q], normalize_embeddings=True, convert_to_numpy=True, device="cuda").astype("float32")
    _, idxs = index.search(q_vec, 3)
    ctxs = [all_chunks[i] for i in idxs[0] if i != -1]
    print(f"\nQ: {q}")
    print(f"A: {local_answer(q, ctxs)[:400]}")


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: Quels sont les delais de prescription en matiere civile ?


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

A: En matière civile au niveau Tunisie, le délai de prescription est généralement d'un an à compter du moment où une obligation contractuelle n'est pas rempliée par un défaut manifeste ou répétitif.

Cependant, il existe des exceptions pour certains types spéciaux :

1) Pour toutes actions fondées sur la faute (délai : cinq ans)
   - Exemples : torts personnels, contrats défaillants 

2) Les obligati

Q: Comment contester une decision administrative ?


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Pour contester une décision administrative en Tunisie, il existe plusieurs voies de recours selon le type d'administration concernée :

1) Pour les décisions administratives des organismes publics ou privés qui n'appartiennent pas à un service spécifique comme celle du travail :
   Il faut saisir directement la Cour Administrative Délégataire dans sa compétence générale pour toutes questions relat

Q: ما هي حقوق المتهم في مرحلة الاستجواب ؟


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


A: وفقًا لأجزاء التشريع التونسي الوارد، فإن بعض الحقوق الأساسية للأُمم المتهمين خلال فترة الاستجواب (التعرف) تتضمن النقاط الرئيسية التالية بنظر لي:

1. **حقوق متهم الشرطي**: 
   - يتميز المتهم بإمكانية استخدام خدمات محاميه.
   - يستطيع الحصول على شهادة الشهود الذين تم التقائهم عبر الهاتف.

2. ** confidentiality**:
    - يستخدم "وسائل التواصل الحديث" مثل الرسائل المصورة والمسمومة لتوفير البيئة المناسب


---
## SECTION D - Save LoRA, Merge, and Push to HuggingFace Hub


In [21]:
# Save LoRA adapters to Drive (small, fast)
model.save_pretrained(str(LORA_DIR))
tokenizer.save_pretrained(str(LORA_DIR))
print(f"LoRA adapters saved -> {LORA_DIR}")
print(f"  Size: {sum(f.stat().st_size for f in LORA_DIR.rglob('*') if f.is_file())/1e6:.1f} MB")


LoRA adapters saved -> /content/drive/MyDrive/LegalMind/mizan-lora
  Size: 131.2 MB


In [22]:
# Merge LoRA into full weights and push (~5-10 min, ~6GB upload)
print(f"Merging and pushing to: https://huggingface.co/{MODEL_REPO}")
print("This takes ~5-10 min...")
model.push_to_hub_merged(
    MODEL_REPO, tokenizer,
    save_method = "merged_16bit",
    token       = HF_TOKEN,
)
print(f"\nModel live at: https://huggingface.co/{MODEL_REPO}")


Merging and pushing to: https://huggingface.co/rayenrayenrayenrayen/legalmind-chatbot
This takes ~5-10 min...


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   0%|          | 4.20MB / 2.20GB            


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:47<00:00, 113.89s/it]


Unsloth: Merge process complete. Saved to `/content/rayenrayenrayenrayen/legalmind-chatbot`

Model live at: https://huggingface.co/rayenrayenrayenrayen/legalmind-chatbot


In [23]:
# SECTION E - LEGAL MIND Gradio App (Pure Python + Gradio for Colab)
# Run Sections A–C first so that embedder, index, all_chunks,
# lang_of, retrieve_raw, local_answer, model, tokenizer are in kernel.

import base64
import calendar
import html
import math
import re
from datetime import date, datetime, timedelta
from pathlib import Path
from uuid import uuid4

import gradio as gr

# ── Gradio version detection ─────────────────────────────────────────────────
_gr_major = int(gr.__version__.split(".")[0])
print(f"Gradio {gr.__version__}  (major = {_gr_major})")


# ── Backend wrappers ──────────────────────────────────────────────────────────
# Define answer() and retrieve() unconditionally so they always point to the
# correct RAG pipeline regardless of cell execution order.

def answer(query, history=None, k=3):
    """Retrieve + generate using the fine-tuned model and FAISS index."""
    lang = lang_of(query)
    try:
        raw = retrieve_raw(query, k=k)          # [(score, chunk), ...]
        contexts = []
        for score, chunk in raw:
            c = dict(chunk)
            c["score"]   = float(score)
            c["excerpt"] = c.get("text", "")[:200]
            contexts.append(c)
    except Exception:
        fb = "عذرا، تعذّر البحث في المصادر." if lang == "ar" \
             else "Désolé, impossible d'accéder aux sources."
        return {"answer": fb, "sources": [], "language": lang, "error": None}
    try:
        text = local_answer(query, contexts)
    except Exception:
        fb = "عذرا، حدث خطأ أثناء توليد الإجابة." if lang == "ar" \
             else "Désolé, une erreur est survenue lors de la génération."
        return {"answer": fb, "sources": contexts, "language": lang, "error": None}
    return {"answer": text, "sources": contexts, "language": lang, "error": None}


def retrieve(query, k=3):
    return [chunk for _, chunk in retrieve_raw(query, k=k)]


# ── Constants ─────────────────────────────────────────────────────────────────
MAX_CHARACTERS = 1000
CHAT_TOP_K     = 3

PLACEHOLDERS = [
    "اطرح سؤالاً قانونيًا...",
    "Posez votre question juridique...",
]

AR_EXAMPLES = [
    "ما هي شروط عقد العمل في تونس؟",
    "ما هي حقوق المتهم في مرحلة الاستجواب؟",
    "كيف يتم تأسيس شركة ذات مسؤولية محدودة؟",
]

FR_EXAMPLES = [
    "Quelles sont les conditions de validité d'un contrat de bail ?",
    "Comment contester une décision administrative ?",
    "Quels sont les délais de prescription en matière civile ?",
]

HEIR_ORDER = [
    "husband","wife","father","mother","son","daughter",
    "sons_son","sons_daughter","full_brother","full_sister",
    "paternal_half_brother","paternal_half_sister",
    "maternal_half_brother","maternal_half_sister",
    "paternal_grandfather","paternal_grandmother","maternal_grandmother",
]

HEIR_LABELS = {
    "husband":               {"fr": "Époux",                      "ar": "الزوج"},
    "wife":                  {"fr": "Épouse",                     "ar": "الزوجة"},
    "son":                   {"fr": "Fils",                       "ar": "الابن"},
    "daughter":              {"fr": "Fille",                      "ar": "البنت"},
    "father":                {"fr": "Père",                       "ar": "الأب"},
    "mother":                {"fr": "Mère",                       "ar": "الأم"},
    "paternal_grandfather":  {"fr": "Grand-père paternel",        "ar": "الجد لأب"},
    "paternal_grandmother":  {"fr": "Grand-mère paternelle",      "ar": "الجدة لأب"},
    "maternal_grandmother":  {"fr": "Grand-mère maternelle",      "ar": "الجدة لأم"},
    "full_brother":          {"fr": "Frère germain",              "ar": "الأخ الشقيق"},
    "full_sister":           {"fr": "Sœur germaine",              "ar": "الأخت الشقيقة"},
    "paternal_half_brother": {"fr": "Frère consanguin",           "ar": "الأخ لأب"},
    "paternal_half_sister":  {"fr": "Sœur consanguine",          "ar": "الأخت لأب"},
    "maternal_half_brother": {"fr": "Frère utérin",              "ar": "الأخ لأم"},
    "maternal_half_sister":  {"fr": "Sœur utérine",              "ar": "الأخت لأم"},
    "sons_son":              {"fr": "Petit-fils (fils du fils)",  "ar": "ابن الابن"},
    "sons_daughter":         {"fr": "Petite-fille (fille du fils)","ar": "بنت الابن"},
}

ARTICLE_REF = {
    "husband":"CSP Art. 143","wife":"CSP Art. 143",
    "son":"CSP Art. 119","daughter":"CSP Art. 118/119",
    "father":"CSP Art. 130","mother":"CSP Art. 131",
    "paternal_grandfather":"CSP Art. 136","paternal_grandmother":"CSP Art. 137",
    "maternal_grandmother":"CSP Art. 137","full_brother":"CSP Art. 136",
    "full_sister":"CSP Art. 136","paternal_half_brother":"CSP Art. 136",
    "paternal_half_sister":"CSP Art. 136",
    "maternal_half_brother":"CSP Art. 131/136","maternal_half_sister":"CSP Art. 131/136",
    "sons_son":"CSP Art. 119","sons_daughter":"CSP Art. 118/119",
}

CLAIM_LABELS = {
    "civil_personal_general": {"fr": "Action personnelle générale",   "ar": "دعوى شخصية عامة"},
    "civil_liability":        {"fr": "Action en responsabilité civile","ar": "دعوى المسؤولية المدنية"},
    "civil_commercial":       {"fr": "Action commerciale",             "ar": "دعوى تجارية"},
    "civil_rent_payment":     {"fr": "Action en paiement de loyer",    "ar": "دعوى خلاص معلوم الكراء"},
    "civil_contract_nullity": {"fr": "Action en nullité de contrat",   "ar": "دعوى بطلان العقد"},
    "civil_alimony":          {"fr": "Créances alimentaires",          "ar": "ديون النفقة"},
    "penal_contravention":    {"fr": "Infraction contraventionnelle",  "ar": "مخالفة"},
    "penal_delit":            {"fr": "Infraction délictuelle",         "ar": "جنحة"},
    "penal_crime":            {"fr": "Crime",                          "ar": "جناية"},
}

COURT_LABELS = {
    "first_instance": {"fr": "Tribunal de première instance", "ar": "المحكمة الابتدائية"},
    "cour_appel":     {"fr": "Cour d'appel",                  "ar": "محكمة الاستئناف"},
    "cour_cassation": {"fr": "Cour de cassation",             "ar": "محكمة التعقيب"},
}

MATTER_LABELS = {
    "civil":         {"fr": "Civil",         "ar": "مدني"},
    "penal":         {"fr": "Pénal",         "ar": "جزائي"},
    "commercial":    {"fr": "Commercial",    "ar": "تجاري"},
    "administratif": {"fr": "Administratif", "ar": "إداري"},
}

JUDGMENT_NATURE_LABELS = {
    "contradictoire":  {"fr": "Contradictoire",    "ar": "حضوري"},
    "par_defaut":      {"fr": "Par défaut",         "ar": "غيابي"},
    "avant_dire_droit":{"fr": "Avant dire droit",  "ar": "قبل الفصل في الأصل"},
}

NOTIFICATION_LABELS = {
    "direct_party": {"fr": "Signification à la partie", "ar": "تبليغ للطرف"},
    "parquet":      {"fr": "Signification à parquet",   "ar": "تبليغ للنيابة"},
    "greffe":       {"fr": "Dépôt au greffe",           "ar": "إيداع بالكتابة"},
}

CIVIL_CLAIMS = [
    "civil_personal_general","civil_liability","civil_commercial",
    "civil_rent_payment","civil_contract_nullity","civil_alimony",
]
PENAL_CLAIMS = ["penal_crime","penal_delit","penal_contravention"]

PRESCRIPTION_RULES = {
    "civil_personal_general": {"years":10,"articleRef":"COC - Action personnelle générale"},
    "civil_liability":        {"years":3, "articleRef":"COC - Responsabilité civile"},
    "civil_commercial":       {"years":5, "articleRef":"Code de commerce - Action commerciale"},
    "civil_rent_payment":     {"years":3, "articleRef":"COC - Paiement de loyer"},
    "civil_contract_nullity": {"years":5, "articleRef":"COC - Nullité du contrat"},
    "civil_alimony":          {"years":1, "articleRef":"CSP - Créances alimentaires"},
    "penal_contravention":    {"years":1, "articleRef":"CPP Art. 5"},
    "penal_delit":            {"years":3, "articleRef":"CPP Art. 5"},
    "penal_crime":            {"years":10,"articleRef":"CPP Art. 5"},
}

LUNAR_HOLIDAY_MAP = {
    2024: {"aidFitr":[(4,10),(4,11)],"aidAdha":[(6,16),(6,17)],"arafat":(6,15)},
    2025: {"aidFitr":[(3,30),(3,31)],"aidAdha":[(6,6),(6,7)],  "arafat":(6,5)},
    2026: {"aidFitr":[(3,20),(3,21)],"aidAdha":[(5,27),(5,28)],"arafat":(5,26)},
    2027: {"aidFitr":[(3,10),(3,11)],"aidAdha":[(5,17),(5,18)],"arafat":(5,16)},
    2028: {"aidFitr":[(2,27),(2,28)],"aidAdha":[(5,5),(5,6)],  "arafat":(5,4)},
}

APPEL_CIVIL_TEMPLATE    = {"name":"Appel","nameAr":"الاستئناف","articleRef":"CPC - Recours en appel","days":30,"months":None}
OPPOSITION_TEMPLATE     = {"name":"Opposition","nameAr":"الاعتراض","articleRef":"CPC - Jugement par défaut","days":15,"months":None}
APPEL_PENAL_TEMPLATE    = {"name":"Appel","nameAr":"الاستئناف","articleRef":"CPP - Voie d'appel pénale","days":10,"months":None}
APPEL_ADMIN_TEMPLATE    = {"name":"Appel","nameAr":"الاستئناف","articleRef":"Contentieux administratif","days":None,"months":2}
CASSATION_CIVIL_TEMPLATE= {"name":"Pourvoi en cassation","nameAr":"الطعن بالتعقيب","articleRef":"CPC - Cassation civile","days":60,"months":None}
CASSATION_PENAL_TEMPLATE= {"name":"Pourvoi en cassation","nameAr":"الطعن بالتعقيب","articleRef":"CPP - Cassation pénale","days":10,"months":None}

ARABIC_RE       = re.compile(r"[\u0600-\u06FF]")
LEGAL_REF_RE    = re.compile(r"(Article\s+\d+|الفصل\s+\d+)", re.IGNORECASE)


# ── Logo ──────────────────────────────────────────────────────────────────────
LOGO_PATH_CANDIDATES = [
    Path("frontend/public/logo-gavel.png"),
    Path("public/logo-gavel.png"),
    Path("logo-gavel.png"),
]

def load_logo_html():
    for p in LOGO_PATH_CANDIDATES:
        if p.exists():
            enc = base64.b64encode(p.read_bytes()).decode()
            return f"<img class='brand-logo' src='data:image/png;base64,{enc}' alt='logo'/>"
    return "<span class='brand-fallback'>⚖️</span>"

LOGO_HTML = load_logo_html()


# ── CSS ───────────────────────────────────────────────────────────────────────
CSS = """
:root {
  --bg-main:    #FDF5E6;
  --bg-card:    #FFFFFF;
  --bg-dark:    #1E0E04;
  --bg-mid:     #3D1E08;
  --text-dark:  #1E0E04;
  --text-light: #FFF8EA;
  --text-muted: #7A4B1E;
  --accent:     #D4A050;
  --accent-soft:#E7C98E;
  --border-gold:rgba(212,160,80,0.30);
  --success:    #1F6B46;
  --warning:    #A65A00;
  --danger:     #8F1D1D;
  --info:       #245C9A;
}

/* ── Base ── */
.gradio-container {
  background: linear-gradient(180deg,#FFFDF8 0%,var(--bg-main) 48%,#F6EBDD 100%) !important;
  color: var(--text-dark) !important;
}

/* ── Force text visibility ── */
.gradio-container, .gradio-container * {
  color: var(--text-dark);
}
.gradio-container label > span,
.gradio-container .label-wrap span,
.gradio-container .block > label > span {
  color: var(--text-dark) !important;
  font-weight: 600;
}

/* ── Tabs ── */
.tab-nav button {
  color: var(--text-dark) !important;
  font-weight: 600;
  background: transparent !important;
}
.tab-nav button.selected {
  color: var(--accent) !important;
  border-bottom: 2px solid var(--accent) !important;
  background: transparent !important;
}

/* ── Inputs & textareas ── */
.gradio-container input,
.gradio-container textarea,
.gradio-container select {
  color: var(--text-dark) !important;
  background: #fff !important;
  border: 1px solid var(--border-gold) !important;
  border-radius: 8px !important;
}
.gradio-container input::placeholder,
.gradio-container textarea::placeholder {
  color: #B07A45 !important;
  opacity: 1 !important;
}
/* Make the chat textarea taller and cleaner */
#chat_input textarea {
  min-height: 52px !important;
  max-height: 160px !important;
  resize: none !important;
  padding: 10px 12px !important;
  font-size: 14px !important;
  line-height: 1.5 !important;
  color: var(--text-dark) !important;
  background: #fff !important;
}

/* ── Buttons ── */
.gradio-container button.primary,
#send_btn button {
  background: var(--accent) !important;
  color: #fff !important;
  border: none !important;
  font-weight: 700 !important;
}
.gradio-container button.secondary {
  background: transparent !important;
  color: var(--text-dark) !important;
  border: 1px solid var(--border-gold) !important;
}
.gradio-container button.stop {
  background: #fee2e2 !important;
  color: var(--danger) !important;
}
.gradio-container button {
  color: var(--text-dark) !important;
}

/* ── Dropdowns ── */
.gradio-container .wrap,
.gradio-container .secondary-wrap,
.gradio-container ul.options {
  background: #fff !important;
}
.gradio-container .item,
.gradio-container li.item {
  color: var(--text-dark) !important;
}
.gradio-container .multiselect span,
.gradio-container .selected-items span {
  color: var(--text-dark) !important;
}

/* ── Radio / Checkbox ── */
.gradio-container .wrap label span,
.gradio-container .radio-group label span,
.gradio-container .checkbox-group label span {
  color: var(--text-dark) !important;
}

/* ── Dataframe ── */
.gradio-container table {
  color: var(--text-dark) !important;
  background: #fff !important;
}
.gradio-container table td,
.gradio-container table th {
  color: var(--text-dark) !important;
  background: #fff !important;
  border-color: var(--border-gold) !important;
}

/* ── Accordion ── */
.gradio-container .label-wrap {
  color: var(--text-dark) !important;
}

/* ── Chat header ── */
#chat-header {
  display: flex;
  align-items: center;
  justify-content: space-between;
  padding: 12px 16px;
  border: 1px solid var(--border-gold);
  border-radius: 12px;
  background: linear-gradient(90deg, rgba(30,14,4,0.98), rgba(61,30,8,0.96));
  color: var(--text-light) !important;
  margin-bottom: 8px;
}
#chat-header * { color: var(--text-light) !important; }

.brand-wrap   { display:flex; align-items:center; gap:10px; }
.brand-logo   { width:34px; height:34px; object-fit:contain; }
.brand-fallback { font-size:24px; line-height:1; }
.brand-title  {
  font-family: "Cascadia Mono","JetBrains Mono",monospace;
  letter-spacing: 0.12em;
  font-size: 14px;
  color: var(--accent-soft) !important;
}

/* ── Sidebar card ── */
.chat-side-card {
  border: 1px solid var(--border-gold);
  border-radius: 12px;
  background: linear-gradient(180deg,rgba(30,14,4,0.98),rgba(61,30,8,0.96));
  padding: 12px;
}
.chat-side-card, .chat-side-card * { color: var(--text-light) !important; }

/* ── Chat display area ── */
#chat_messages {
  min-height: 520px;
  max-height: 520px;
  overflow-y: auto;
  border: 1px solid var(--border-gold);
  border-radius: 12px;
  background: #FAF6EF;
  padding: 8px 10px;
}
/* Hide Gradio's own label for the HTML component */
#chat_messages_container > .label-wrap { display: none !important; }

/* ── Message bubbles ── */
.msg-row { width:100%; display:flex; margin:8px 0; }
.msg-row.user.ar  { justify-content:flex-end; }
.msg-row.user.fr  { justify-content:flex-start; }
.msg-row.assistant{ justify-content:flex-start; }

.msg-bubble {
  max-width: 86%;
  border-radius: 18px;
  padding: 10px 14px;
  box-shadow: 0 4px 12px rgba(30,14,4,0.07);
  line-height: 1.65;
  font-size: 14px;
}

.msg-user {
  background: linear-gradient(145deg,rgba(30,14,4,0.97),rgba(61,30,8,0.97));
}
.msg-user, .msg-user * { color: var(--text-light) !important; }
.msg-user.ar { border-radius:18px 18px 18px 4px; text-align:right; direction:rtl; }
.msg-user.fr { border-radius:18px 18px 4px 18px; text-align:left;  direction:ltr; }

.msg-assistant {
  background: var(--bg-card);
  border: 1px solid var(--border-gold);
  border-radius: 18px 18px 18px 4px;
}
.msg-assistant, .msg-assistant * { color: var(--text-dark) !important; }

.msg-meta {
  display:flex; gap:8px; margin-top:6px;
  font-size:11px; align-items:center;
}
.msg-user .msg-meta * { color: rgba(255,248,234,0.75) !important; }
.msg-assistant .msg-meta * { color: #6A4A2D !important; }

.lang-badge {
  display:inline-flex; border-radius:999px;
  border:1px solid rgba(36,92,154,0.3);
  background:rgba(36,92,154,0.11);
  padding:2px 8px; font-size:11px; font-weight:700;
}
.msg-user .lang-badge { border-color:rgba(255,248,234,0.3); background:rgba(255,248,234,0.15); }
.msg-user .lang-badge * { color: rgba(255,248,234,0.9) !important; }
.msg-assistant .lang-badge * { color: var(--info) !important; }

.warning-banner {
  margin-top:8px; border:1px solid rgba(166,90,0,0.35);
  background:rgba(166,90,0,0.12); border-radius:8px;
  padding:8px; font-size:12px;
}
.warning-banner * { color: var(--warning) !important; }

.legal-ref {
  background:rgba(212,160,80,0.2); border-radius:6px;
  padding:1px 4px;
  font-family:"Cascadia Mono","JetBrains Mono",monospace;
}

/* ── Thinking animation ── */
.thinking { display:flex; align-items:center; gap:5px; }
.think-dot {
  width:7px; height:7px; border-radius:999px;
  background:var(--accent);
  animation:think-wave 1s infinite ease-in-out;
}
.think-dot:nth-child(2){animation-delay:0.15s;}
.think-dot:nth-child(3){animation-delay:0.3s;}
@keyframes think-wave {
  0%,80%,100%{transform:translateY(0);opacity:0.55;}
  40%{transform:translateY(-5px);opacity:1;}
}

/* ── Sources panel ── */
.source-panel-empty {
  border:1px dashed var(--border-gold); border-radius:10px;
  padding:12px; font-size:13px;
}
.source-panel-empty * { color: var(--text-muted) !important; }

.source-card {
  border:1px solid var(--border-gold); border-radius:12px;
  background:var(--bg-card); padding:10px; margin-bottom:8px;
}
.source-card * { color: var(--text-dark) !important; }
.source-title { font-weight:700; margin-bottom:2px; }
.source-sub   { font-size:12px; margin-bottom:6px; }
.source-sub * { color: var(--text-muted) !important; }

.score-track  { width:100%; height:7px; background:rgba(212,160,80,0.2); border-radius:999px; overflow:hidden; }
.score-fill   { height:7px; }
.score-good   { background:rgba(31,107,70,0.8); }
.score-mid    { background:rgba(166,90,0,0.8); }
.score-low    { background:rgba(143,29,29,0.8); }
.score-label  { margin-top:6px; font-size:12px; font-weight:600; }
.score-label * { color: var(--text-muted) !important; }

/* ── Calc cards ── */
.metric-card { border:1px solid var(--border-gold); border-radius:12px; background:var(--bg-card); padding:10px; }
.metric-card * { color: var(--text-dark) !important; }

.badge { display:inline-flex; border-radius:999px; padding:3px 10px; font-size:12px; font-weight:700; border:1px solid transparent; }
.badge-success { border-color:rgba(31,107,70,0.32);  background:rgba(31,107,70,0.11);  }
.badge-success * { color: var(--success) !important; }
.badge-warning { border-color:rgba(166,90,0,0.34);   background:rgba(166,90,0,0.13);   }
.badge-warning * { color: var(--warning) !important; }
.badge-danger  { border-color:rgba(143,29,29,0.35);  background:rgba(143,29,29,0.13);  }
.badge-danger  * { color: var(--danger) !important;  }
.badge-neutral { border-color:var(--border-gold);    background:rgba(212,160,80,0.07); }
.badge-neutral * { color: var(--text-dark) !important; }

.soft-card { border:1px solid var(--border-gold); border-radius:12px; background:rgba(212,160,80,0.07); padding:10px; }
.soft-card * { color: var(--text-dark) !important; }
.calc-title { font-weight:700; margin-bottom:6px; }

/* ── Examples ── */
.empty-shell { border:1px solid var(--border-gold); border-radius:14px; background:var(--bg-card); padding:14px; }
.empty-shell * { color: var(--text-dark) !important; }

.meta-chip {
  display:inline-flex; align-items:center; border-radius:999px;
  border:1px solid rgba(212,160,80,0.35); background:rgba(212,160,80,0.16);
  padding:4px 10px; font-size:12px; font-weight:700;
}
.meta-chip * { color: var(--text-dark) !important; }

.char-counter { text-align:right; font-size:12px; margin-top:4px; }
.char-counter * { color: var(--text-muted) !important; }
"""

# ── Head JS ───────────────────────────────────────────────────────────────────
HEAD_JS = """
<script>
(function () {
  const placeholders = ["اطرح سؤالاً قانونيًا...", "Posez votre question juridique..."];

  function attachInputBehaviors() {
    const wrapper = document.getElementById("chat_input");
    if (!wrapper) return;
    const ta = wrapper.querySelector("textarea");
    if (!ta || ta.dataset.lmBound) return;
    ta.dataset.lmBound = "1";

    const autoSize = () => {
      ta.style.height = "auto";
      ta.style.height = Math.min(ta.scrollHeight, 160) + "px";
    };
    const updateDir = () => {
      const txt = ta.value || "";
      const ar = (txt.match(/[\u0600-\u06FF]/g) || []).length;
      const isAr = txt.trim().length > 0 && (ar / txt.length) > 0.25;
      ta.dir = isAr ? "rtl" : "ltr";
      ta.style.textAlign = isAr ? "right" : "left";
    };
    ta.addEventListener("input", () => { autoSize(); updateDir(); });
    ta.addEventListener("keydown", (e) => {
      if (e.key === "Enter" && !e.shiftKey) {
        e.preventDefault();
        const btnWrap = document.getElementById("send_btn");
        if (!btnWrap) return;
        const btn = btnWrap.querySelector("button");
        if (btn && !btn.disabled) btn.click();
      }
    });
    autoSize(); updateDir();
  }

  function rotatePlaceholder() {
    const wrapper = document.getElementById("chat_input");
    if (!wrapper) return;
    const ta = wrapper.querySelector("textarea");
    if (!ta) return;
    let idx = 0;
    setInterval(() => {
      if (!ta.value) {
        ta.placeholder = placeholders[idx % placeholders.length];
        idx++;
      }
    }, 4000);
  }

  function scrollChatToBottom() {
    const el = document.getElementById("chat_messages");
    if (el) el.scrollTop = el.scrollHeight;
  }

  // Poll until the textarea is mounted
  let initialized = false;
  const tick = setInterval(() => {
    attachInputBehaviors();
    scrollChatToBottom();
    if (!initialized) { rotatePlaceholder(); initialized = true; }
  }, 600);
  setTimeout(() => clearInterval(tick), 120000);

  // Also scroll on DOM mutations in the chat area
  const observer = new MutationObserver(scrollChatToBottom);
  const waitForChat = setInterval(() => {
    const el = document.getElementById("chat_messages");
    if (el) {
      observer.observe(el, { childList: true, subtree: true });
      clearInterval(waitForChat);
    }
  }, 800);
})();
</script>
"""


# ── Generic helpers ───────────────────────────────────────────────────────────
def today_string():
    return datetime.now().strftime("%Y-%m-%d")

def parse_date(value):
    if value is None: return None
    if isinstance(value, datetime): return value.date()
    if isinstance(value, date): return value
    text = str(value).strip()
    if not text: return None
    try: return datetime.fromisoformat(text.replace("Z","")).date()
    except Exception: pass
    for fmt in ["%Y-%m-%d","%Y/%m/%d","%d/%m/%Y","%Y-%m-%d %H:%M:%S","%Y-%m-%dT%H:%M:%S"]:
        try: return datetime.strptime(text, fmt).date()
        except Exception: continue
    return None

def fmt_date(value):
    return value.strftime("%Y-%m-%d") if value else "-"

def add_years(base, years):
    try: return base.replace(year=base.year + years)
    except ValueError: return base.replace(month=2, day=28, year=base.year + years)

def add_days(value, days): return value + timedelta(days=days)

def add_months(value, months):
    month_idx = value.month - 1 + months
    year  = value.year + month_idx // 12
    month = month_idx % 12 + 1
    last_day = calendar.monthrange(year, month)[1]
    return date(year, month, min(value.day, last_day))

def days_between(start, end): return (end - start).days

def get_tunisian_public_holidays(year):
    fixed = [
        date(year,3,20),date(year,3,21),date(year,4,9),date(year,5,1),
        date(year,7,25),date(year,8,13),date(year,10,15),date(year,11,7),
    ]
    lunar = LUNAR_HOLIDAY_MAP.get(year)
    if not lunar: return fixed
    return (
        fixed
        + [date(year,m,d) for (m,d) in lunar["aidFitr"]]
        + [date(year,m,d) for (m,d) in lunar["aidAdha"]]
        + [date(year, lunar["arafat"][0], lunar["arafat"][1])]
    )

def is_working_day(value):
    if value.weekday() in (4,5): return False
    return value not in get_tunisian_public_holidays(value.year)

def adjust_to_next_working_day(value):
    while not is_working_day(value): value += timedelta(days=1)
    return value

def detect_input_language(text):
    if not text or not str(text).strip(): return "fr"
    ar = len(ARABIC_RE.findall(str(text)))
    return "ar" if ar / max(len(str(text)),1) > 0.25 else "fr"

def highlight_legal_references(text):
    raw = text or ""
    chunks, cursor = [], 0
    for m in LEGAL_REF_RE.finditer(raw):
        chunks.append(html.escape(raw[cursor:m.start()]))
        chunks.append(f"<span class='legal-ref'>{html.escape(m.group(0))}</span>")
        cursor = m.end()
    chunks.append(html.escape(raw[cursor:]))
    return "".join(chunks).replace("\n","<br>")

def format_currency(value):
    if value is None: return "-"
    try: n = float(value)
    except Exception: return "-"
    if not math.isfinite(n): return "-"
    return f"{n:,.3f} TND".replace(",", "\u202f")

def status_badge_html(status):
    if status == "expired": return "<span class='badge badge-danger'>Expiré</span>"
    if status == "warning": return "<span class='badge badge-warning'>Alerte</span>"
    return "<span class='badge badge-success'>Actif</span>"

def urgent_badge_html(days_remaining):
    if days_remaining is None: return "<span class='badge badge-neutral'>-</span>"
    if days_remaining < 5:  return f"<span class='badge badge-danger'>{days_remaining} j</span>"
    if days_remaining <= 15:return f"<span class='badge badge-warning'>{days_remaining} j</span>"
    return f"<span class='badge badge-success'>{days_remaining} j</span>"


# ── Chat state ────────────────────────────────────────────────────────────────
def new_conversation():
    now = datetime.now()
    return {"id":str(uuid4()),"messages":[],"created_at":now,"updated_at":now}

def init_chat_state():
    conv = new_conversation()
    return {"conversations":[conv],"active_id":conv["id"],"latest_sources":[]}

def ensure_chat_state(state):
    if not isinstance(state, dict): return init_chat_state()
    if not state.get("conversations"): return init_chat_state()
    if "active_id" not in state: state["active_id"] = state["conversations"][0]["id"]
    return state

def sort_conversations(state):
    state["conversations"].sort(key=lambda c: c["updated_at"], reverse=True)

def get_active_conversation(state):
    state = ensure_chat_state(state)
    for conv in state["conversations"]:
        if conv["id"] == state["active_id"]: return conv
    state["active_id"] = state["conversations"][0]["id"]
    return state["conversations"][0]

def conversation_lang(conversation):
    for msg in conversation["messages"]:
        if msg["role"] == "user": return msg["lang"]
    return "fr"

def conversation_title(conversation):
    for msg in conversation["messages"]:
        if msg["role"] == "user":
            t = msg["content"]
            return t[:40] + ("..." if len(t) > 40 else "")
    return "Nouvelle conversation / محادثة جديدة"

def relative_time_label(ts, lang):
    diff    = datetime.now() - ts
    minutes = max(1, int(diff.total_seconds() // 60))
    if minutes < 60:
        return f"منذ {minutes} دقيقة" if lang == "ar" else f"il y a {minutes} min"
    hours = minutes // 60
    if hours < 24:
        if lang == "ar":
            if hours == 1: return "منذ ساعة"
            if hours == 2: return "منذ ساعتين"
            return f"منذ {hours} ساعات"
        return f"il y a {hours} h"
    days = hours // 24
    if lang == "ar":
        if days == 1: return "منذ يوم"
        if days == 2: return "منذ يومين"
        return f"منذ {days} أيام"
    return f"il y a {days} j"

def build_sidebar_choices(state):
    sort_conversations(state)
    choices = []
    for conv in state["conversations"]:
        lang  = conversation_lang(conv)
        label = f"{conversation_title(conv)} • {relative_time_label(conv['updated_at'], lang)}"
        choices.append((label, conv["id"]))
    return choices

def build_turn_history(messages):
    turns, pending = [], None
    for msg in messages:
        if msg["role"] == "thinking": continue
        if msg["role"] == "user": pending = msg["content"]; continue
        if msg["role"] == "assistant" and pending:
            turns.append({"user": pending, "assistant": msg["content"]})
            pending = None
    return turns[-10:]

def normalize_sources(raw):
    out = []
    if not isinstance(raw, list): return out
    for item in raw:
        if not isinstance(item, dict): continue
        try: score = float(item.get("score", 0.0))
        except Exception: score = 0.0
        out.append({
            "title":   str(item.get("title") or item.get("source") or "Document sans titre"),
            "source":  str(item.get("source") or "Source"),
            "article": str(item.get("article") or ""),
            "url":     str(item.get("url") or ""),
            "score":   score,
            "excerpt": str(item.get("excerpt") or ""),
        })
    return out

def make_message(role, content, lang, sources=None, warning=None):
    return {
        "id":        str(uuid4()),
        "role":      role,
        "content":   content,
        "lang":      lang,
        "sources":   sources or [],
        "warning":   warning or "",
        "timestamp": datetime.now(),
    }

def message_meta_html(msg):
    lang = msg.get("lang","fr")
    ts   = msg["timestamp"].strftime("%H:%M")
    return f"<div class='msg-meta'><span class='lang-badge'>{lang.upper()}</span><span>{ts}</span></div>"

def render_message_html(msg):
    role = msg.get("role","assistant")
    lang = msg.get("lang","fr")

    if role == "thinking":
        txt = "جار البحث في النصوص..." if lang == "ar" else "Recherche dans les textes..."
        return (
            "<div class='msg-row assistant'>"
            "<div class='msg-bubble msg-assistant'>"
            "<div class='thinking'>"
            "<span class='think-dot'></span><span class='think-dot'></span><span class='think-dot'></span>"
            "</div>"
            f"<div style='margin-top:8px;font-size:12px'>{html.escape(txt)}</div>"
            "</div></div>"
        )

    body = highlight_legal_references(msg.get("content",""))
    meta = message_meta_html(msg)

    if role == "user":
        ac = "ar" if lang == "ar" else "fr"
        return (
            f"<div class='msg-row user {ac}'>"
            f"<div class='msg-bubble msg-user {ac}'>"
            f"<div>{body}</div>{meta}"
            "</div></div>"
        )

    warning = (msg.get("warning") or "").strip()
    warn_html = (
        f"<div class='warning-banner'>{html.escape(warning).replace(chr(10),'<br>')}</div>"
        if warning else ""
    )
    return (
        "<div class='msg-row assistant'>"
        "<div class='msg-bubble msg-assistant'>"
        f"<div>{body}</div>{warn_html}{meta}"
        "</div></div>"
    )

# ── Chat HTML renderer (replaces gr.Chatbot) ──────────────────────────────────
def render_chat_html(state):
    conv = get_active_conversation(state)
    if not conv["messages"]:
        return (
            "<div id='chat_messages' style='display:flex;align-items:center;"
            "justify-content:center;height:100%;'>"
            "<div style='text-align:center;color:#B07A45;font-size:15px;padding:40px'>"
            "⚖️<br>Posez votre question juridique<br>اطرح سؤالك القانوني"
            "</div></div>"
        )
    parts = [render_message_html(m) for m in conv["messages"]]
    return "<div>" + "".join(parts) + "</div>"

def latest_sources_from_messages(messages):
    for msg in reversed(messages):
        if msg.get("role") == "assistant": return msg.get("sources",[]) or []
    return []

def source_card_html(source):
    score = max(0.0, min(1.0, float(source.get("score",0.0))))
    pct   = int(round(score * 100))
    cls   = "score-good" if score > 0.8 else ("score-mid" if score > 0.6 else "score-low")
    title   = html.escape(str(source.get("title") or "Document sans titre"))
    src_name= html.escape(str(source.get("source") or "Source"))
    article = html.escape(str(source.get("article") or ""))
    excerpt = html.escape(str(source.get("excerpt") or ""))
    exc_html = (
        "<details style='margin-top:6px'>"
        "<summary style='cursor:pointer;font-size:12px'>Extrait</summary>"
        f"<div style='margin-top:4px;font-size:12px'>{excerpt}</div>"
        "</details>" if excerpt else ""
    )
    return (
        "<div class='source-card'>"
        f"<div class='source-title'>{title}</div>"
        f"<div class='source-sub'>{src_name}"
        + (f" | {article}" if article else "") + "</div>"
        "<div class='score-track'>"
        f"<div class='score-fill {cls}' style='width:{pct}%'></div>"
        "</div>"
        f"<div class='score-label'>Pertinence : {pct} %</div>"
        f"{exc_html}</div>"
    )

def render_sources_panel(sources):
    if not sources:
        return "<div class='source-panel-empty'>Aucune source pour le moment.</div>"
    return "".join(source_card_html(s) for s in sources)

def render_input_lang_badge(text):
    lang = detect_input_language(text or "")
    return f"<span class='meta-chip'>{lang.upper()}</span>"

def render_counter(text):
    n = len(text or "")
    if n > 300: return f"<div class='char-counter'>{n}/{MAX_CHARACTERS}</div>"
    return "<div class='char-counter'></div>"

def compose_chat_outputs(state):
    state = ensure_chat_state(state)
    conv  = get_active_conversation(state)
    state["latest_sources"] = latest_sources_from_messages(conv["messages"])

    chat_html      = render_chat_html(state)
    radio_update   = gr.update(choices=build_sidebar_choices(state), value=state["active_id"])
    examples_update= gr.update(visible=(len(conv["messages"]) == 0))

    sources = state["latest_sources"]
    sources_acc_update = gr.update(label=f"Sources ({len(sources)})", open=False)
    sources_html   = render_sources_panel(sources)

    return chat_html, state, radio_update, examples_update, sources_acc_update, sources_html

def safe_answer_call(query, history):
    try:
        response = answer(query, history=history, k=CHAT_TOP_K)
        if not isinstance(response, dict):
            raise ValueError("Backend returned non-dict response")
        return response
    except Exception as exc:
        lang = detect_input_language(query)
        fallback = (
            "عذرا، حدث خطأ أثناء معالجة السؤال."
            if lang == "ar" else
            "Désolé, une erreur s'est produite pendant le traitement."
        )
        return {"answer": fallback, "sources": [], "language": lang, "error": None}

def confidence_warning(lang, sources, backend_error):
    if sources:
        avg = sum(float(s.get("score",0.0)) for s in sources) / len(sources)
        if avg < 0.60:
            return (
                "تحذير: قد تكون الإجابة مبنية على مصادر ذات صلة جزئية."
                if lang == "ar" else
                "Avertissement : réponse basée sur des sources partiellement pertinentes."
            )
    return ""

def append_assistant_from_response(conversation, response):
    lang    = "ar" if str(response.get("language","fr")).lower() == "ar" else "fr"
    sources = normalize_sources(response.get("sources",[]))
    warning = confidence_warning(lang, sources, None)
    msg = make_message(
        role="assistant",
        content=str(response.get("answer","")),
        lang=lang,
        sources=sources,
        warning=warning,
    )
    conversation["messages"].append(msg)
    conversation["updated_at"] = datetime.now()
    return msg


# ── Input event handlers ──────────────────────────────────────────────────────
def on_input_change(text):
    text = (text or "")[:MAX_CHARACTERS]
    return text, render_input_lang_badge(text), render_counter(text)

def on_select_conversation(selected_id, state):
    state = ensure_chat_state(state)
    if selected_id:
        for conv in state["conversations"]:
            if conv["id"] == selected_id:
                state["active_id"] = selected_id; break
    return compose_chat_outputs(state)

def on_new_conversation(state):
    state = ensure_chat_state(state)
    conv  = new_conversation()
    state["conversations"].append(conv)
    state["active_id"] = conv["id"]
    sort_conversations(state)
    return compose_chat_outputs(state)

def on_clear_history(state):
    return compose_chat_outputs(init_chat_state())

def on_send_message(query, state):
    state = ensure_chat_state(state)
    query = (query or "").strip()[:MAX_CHARACTERS]

    if not query:
        yield (*compose_chat_outputs(state), "", gr.update(interactive=True))
        return

    conv    = get_active_conversation(state)
    history = build_turn_history(conv["messages"])
    lang    = detect_input_language(query)

    conv["messages"].append(make_message("user",     query, lang))
    conv["messages"].append(make_message("thinking", "",    lang))
    conv["updated_at"] = datetime.now()
    sort_conversations(state)

    yield (*compose_chat_outputs(state), "", gr.update(interactive=False))

    response = safe_answer_call(query, history)

    conv = get_active_conversation(state)
    conv["messages"] = [m for m in conv["messages"] if m.get("role") != "thinking"]
    append_assistant_from_response(conv, response)
    conv["updated_at"] = datetime.now()
    sort_conversations(state)

    yield (*compose_chat_outputs(state), "", gr.update(interactive=True))

def push_summary_to_chat(summary, state):
    state   = ensure_chat_state(state)
    summary = (summary or "").strip()
    if not summary:
        lang = "fr"
        msg  = "Calculez d'abord puis cliquez sur Explain with AI."
        return msg, *compose_chat_outputs(state)

    conv    = get_active_conversation(state)
    history = build_turn_history(conv["messages"])
    lang    = detect_input_language(summary)

    conv["messages"].append(make_message("user", summary, lang))
    response  = safe_answer_call(summary, history)
    assistant = append_assistant_from_response(conv, response)
    conv["updated_at"] = datetime.now()
    sort_conversations(state)

    return assistant["content"], *compose_chat_outputs(state)


# ── Mirath logic ──────────────────────────────────────────────────────────────
def gcd_int(a, b):
    x, y = abs(int(a)), abs(int(b))
    while y: x, y = y, x % y
    return x or 1

def decimal_to_fraction(value):
    D = 12000
    N = int(round(float(value) * D))
    g = gcd_int(N, D)
    return {"numerator": N//g, "denominator": D//g}

def calculate_inheritance(input_data):
    heir_map    = {h["type"]:h for h in input_data["heirs"] if int(h.get("count",0))>0}
    hajb_notes  = []
    blocked_by  = {}

    def count(t): return int(heir_map.get(t,{}).get("count",0))
    def has_any(types): return any(count(t)>0 for t in types)
    def block(blocked, blocker, note):
        if count(blocked)>0 and blocked not in blocked_by:
            blocked_by[blocked]=blocker; hajb_notes.append(note)

    has_descendants = has_any(["son","daughter","sons_son","sons_daughter"])
    has_sons        = count("son")>0
    daughter_count  = count("daughter")
    sons_son_count  = count("sons_son")
    sons_daughter_count = count("sons_daughter")
    phb = count("paternal_half_brother")
    phs = count("paternal_half_sister")

    if count("father")>0:
        for t in ["paternal_grandfather","full_brother","full_sister",
                  "paternal_half_brother","paternal_half_sister",
                  "maternal_half_brother","maternal_half_sister"]:
            block(t,"father",f"Hajb: Father blocks {t}")
    if has_sons:
        for t in ["sons_son","sons_daughter","full_brother","full_sister",
                  "paternal_half_brother","paternal_half_sister"]:
            block(t,"son",f"Hajb: Son blocks {t}")
    if has_descendants:
        block("maternal_half_brother","son","Hajb: Children block maternal half siblings")
        block("maternal_half_sister","son","Hajb: Children block maternal half siblings")
    if count("full_brother")>0:
        block("paternal_half_brother","full_brother","Hajb: Full brother blocks paternal half brother")
        block("paternal_half_sister","full_brother","Hajb: Full brother blocks paternal half sister")
    if daughter_count>=2 and sons_son_count==0:
        block("sons_daughter","daughter","Hajb: Two daughters block son's daughter")

    shares = []
    def add_fixed(htype, ratio):
        heir = heir_map.get(htype)
        if (not heir) or (htype in blocked_by): return
        shares.append({"heir":heir,"share":float(ratio),"shareType":"fard","articleRef":ARTICLE_REF[htype]})

    sib_count = sum(count(t) for t in [
        "full_brother","full_sister","paternal_half_brother",
        "paternal_half_sister","maternal_half_brother","maternal_half_sister"])

    if count("husband")>0: add_fixed("husband",(1/4) if has_descendants else (1/2))
    wc = count("wife")
    if wc>0 and "wife" not in blocked_by:
        shares.append({"heir":heir_map["wife"],"share":(1/8) if has_descendants else (1/4),"shareType":"fard","articleRef":ARTICLE_REF["wife"]})
    if count("mother")>0 and "mother" not in blocked_by:
        add_fixed("mother",(1/6) if (has_descendants or sib_count>=2) else (1/3))

    father_present = count("father")>0 and "father" not in blocked_by
    if father_present and has_sons: add_fixed("father",1/6)

    son_count = count("son")
    if daughter_count>0 and son_count==0 and "daughter" not in blocked_by:
        shares.append({"heir":heir_map["daughter"],"share":(1/2) if daughter_count==1 else (2/3),"shareType":"fard","articleRef":ARTICLE_REF["daughter"]})
    if sons_son_count==0 and sons_daughter_count>0 and son_count==0 and "sons_daughter" not in blocked_by:
        shares.append({"heir":heir_map["sons_daughter"],"share":(1/2) if sons_daughter_count==1 else (2/3),"shareType":"fard","articleRef":ARTICLE_REF["sons_daughter"]})

    ubt = count("maternal_half_brother")+count("maternal_half_sister")
    if ubt>0 and not has_descendants and not father_present and "maternal_half_brother" not in blocked_by:
        total_share = (1/6) if ubt==1 else (1/3)
        ubc = count("maternal_half_brother"); usc = count("maternal_half_sister")
        if ubc>0: shares.append({"heir":heir_map["maternal_half_brother"],"share":total_share*(ubc/ubt),"shareType":"fard","articleRef":ARTICLE_REF["maternal_half_brother"]})
        if usc>0: shares.append({"heir":heir_map["maternal_half_sister"],"share":total_share*(usc/ubt),"shareType":"fard","articleRef":ARTICLE_REF["maternal_half_sister"]})

    fixed_sum_raw = sum(r["share"] for r in shares)
    awl_applied = fixed_sum_raw > 1
    if awl_applied:
        for r in shares: r["share"] /= fixed_sum_raw
    fixed_sum = sum(r["share"] for r in shares)
    remainder = max(0.0, 1-fixed_sum)

    asaba = []
    if remainder > 0:
        if son_count>0:
            units = son_count*2 + daughter_count
            asaba.append({"heir":heir_map["son"],"share":remainder*((son_count*2)/units),"shareType":"asaba","articleRef":ARTICLE_REF["son"]})
            if daughter_count>0:
                asaba.append({"heir":heir_map["daughter"],"share":remainder*(daughter_count/units),"shareType":"asaba","articleRef":ARTICLE_REF["daughter"]})
            remainder = 0
        elif father_present:
            asaba.append({"heir":heir_map["father"],"share":remainder,"shareType":"asaba","articleRef":ARTICLE_REF["father"]}); remainder=0
        else:
            fbc,fsc = count("full_brother"),count("full_sister")
            if fbc>0 or fsc>0:
                units = fbc*2+fsc
                if fbc>0 and "full_brother" not in blocked_by: asaba.append({"heir":heir_map["full_brother"],"share":remainder*((fbc*2)/units),"shareType":"asaba","articleRef":ARTICLE_REF["full_brother"]})
                if fsc>0 and "full_sister" not in blocked_by:  asaba.append({"heir":heir_map["full_sister"],"share":remainder*(fsc/units),"shareType":"asaba","articleRef":ARTICLE_REF["full_sister"]})
                remainder=0
            elif phb>0 or phs>0:
                units = phb*2+phs
                if phb>0 and "paternal_half_brother" not in blocked_by: asaba.append({"heir":heir_map["paternal_half_brother"],"share":remainder*((phb*2)/units),"shareType":"asaba","articleRef":ARTICLE_REF["paternal_half_brother"]})
                if phs>0 and "paternal_half_sister" not in blocked_by:  asaba.append({"heir":heir_map["paternal_half_sister"],"share":remainder*(phs/units),"shareType":"asaba","articleRef":ARTICLE_REF["paternal_half_sister"]})
                remainder=0

    radd_applied = False
    if remainder>0 and not asaba and shares:
        eligible = [r for r in shares if r["heir"]["type"] not in ("husband","wife")]
        es = sum(r["share"] for r in eligible)
        if es>0:
            radd_applied=True
            for r in eligible: r["share"] += remainder*(r["share"]/es)
            remainder=0

    all_rows = shares + asaba
    heir_results = []
    for row in all_rows:
        phs2 = row["share"]/row["heir"]["count"]
        frac = decimal_to_fraction(phs2)
        amt  = phs2*input_data.get("estateValue") if input_data.get("estateValue") else None
        heir_results.append({"heir":row["heir"],"shareNumerator":frac["numerator"],"shareDenominator":frac["denominator"],"shareType":row["shareType"],"articleRef":row["articleRef"],"amountTND":amt})

    for heir in input_data["heirs"]:
        if heir.get("count",0)<=0: continue
        blocker = blocked_by.get(heir["type"])
        if not blocker: continue
        heir_results.append({"heir":heir,"shareNumerator":0,"shareDenominator":1,"shareType":"blocked","articleRef":ARTICLE_REF[heir["type"]],"blockedBy":blocker,"amountTND":0})

    total = sum(r["shareNumerator"]/r["shareDenominator"] for r in heir_results if r["shareType"]!="blocked")
    tf    = decimal_to_fraction(total)
    return {"heirs":heir_results,"awlApplied":awl_applied,"raddApplied":radd_applied,"hajbNotes":hajb_notes,"totalVerification":f"{tf['numerator']}/{tf['denominator']}"}


def mirath_calculate(deceased_gender, estate_value, *heir_values):
    counts = {h:max(0,min(20,int(float(v or 0)))) for h,v in zip(HEIR_ORDER,heir_values)}
    if sum(counts.values())==0:
        return [], "<div class='soft-card'>Ajoutez au moins un héritier.</div>","","",""

    estate = None
    if estate_value not in (None,""):
        try: estate = float(estate_value)
        except Exception: return [],"<div class='soft-card'>Valeur invalide.</div>","","",""
        if not math.isfinite(estate) or estate<=0:
            return [],"<div class='soft-card'>Valeur invalide.</div>","","",""

    result = calculate_inheritance({
        "deceasedGender": deceased_gender,
        "estateValue":    estate,
        "heirs": [{"type":h,"count":counts[h]} for h in HEIR_ORDER],
    })

    non_blocked  = [r for r in result["heirs"] if r["shareType"]!="blocked"]
    total_pct    = sum((r["shareNumerator"]/r["shareDenominator"])*100 for r in non_blocked)

    table_rows = []
    for row in result["heirs"]:
        labels   = HEIR_LABELS[row["heir"]["type"]]
        fraction = "0/1" if row["shareType"]=="blocked" else f"{row['shareNumerator']}/{row['shareDenominator']}"
        pct      = 0.0 if row["shareType"]=="blocked" else (row["shareNumerator"]/row["shareDenominator"])*100
        table_rows.append([
            f"{labels['fr']} / {labels['ar']} ×{row['heir']['count']}",
            fraction, f"{pct:.2f}%", format_currency(row.get("amountTND")),
        ])
    total_amount = sum((r.get("amountTND") or 0.0) for r in result["heirs"])
    table_rows.append(["Total / المجموع", result["totalVerification"], f"{total_pct:.2f}%", format_currency(total_amount if estate else None)])

    badges = (
        "<div class='soft-card'>"
        f"<span class='badge badge-neutral'>Awl : {'Oui' if result['awlApplied'] else 'Non'}</span> "
        f"<span class='badge badge-neutral'>Radd : {'Oui' if result['raddApplied'] else 'Non'}</span>"
        "</div>"
    )
    notes_html = ""
    if result["hajbNotes"]:
        notes_html = "<div class='soft-card'><div class='calc-title'>Notes Hajb</div><ul>"
        for n in result["hajbNotes"]: notes_html += f"<li>{html.escape(n)}</li>"
        notes_html += "</ul></div>"

    top = []
    for item in result["heirs"]:
        if item["shareType"]=="blocked": continue
        top.append(f"{HEIR_LABELS[item['heir']['type']]['fr']}: {item['shareNumerator']}/{item['shareDenominator']}")
        if len(top)>=5: break
    summary = f"Explique ce résultat de succession tunisienne ({result['totalVerification']}): {'; '.join(top)}"

    return table_rows, badges, notes_html, summary

def mirath_reset():
    return ["male", None, *[0]*len(HEIR_ORDER), [], "", "", ""]


# ── Prescription logic ────────────────────────────────────────────────────────
def calculate_suspension_days(from_date, expiry_before, suspensions):
    total = 0
    for s in suspensions:
        start, end = s.get("startDate"), s.get("endDate")
        if not start or not end or end<=start: continue
        o_start = max(start, from_date)
        o_end   = min(end, expiry_before)
        if o_end > o_start: total += days_between(o_start, o_end)
    return max(0, total)

def latest_interruption_date(start_date, interruptions):
    valid = sorted([e["date"] for e in interruptions if e.get("date") and e["date"]>=start_date])
    return valid[-1] if valid else start_date

def calculate_prescription(input_data):
    rule  = PRESCRIPTION_RULES[input_data["claimType"]]
    today = date.today()
    orig  = input_data["startDate"]
    adj   = latest_interruption_date(orig, input_data["interruptions"])
    base_expiry = add_years(adj, rule["years"])
    susp_days   = calculate_suspension_days(adj, base_expiry, input_data["suspensions"])
    expiry_raw  = base_expiry + timedelta(days=susp_days)
    expiry      = adjust_to_next_working_day(expiry_raw)
    remaining   = days_between(today, expiry)
    status      = "expired" if remaining<0 else ("warning" if remaining<=90 else "active")
    ic          = len([e for e in input_data["interruptions"] if e.get("date") and e["date"]>=orig])
    return {
        "claimType": input_data["claimType"], "prescriptionYears": rule["years"],
        "articleRef": rule["articleRef"], "startDate": orig, "adjustedStartDate": adj,
        "expiryDate": expiry, "status": status, "daysRemaining": remaining,
        "suspensionNote":    f"Délai prolongé de {susp_days} jour(s) pour suspension(s)." if susp_days>0 else None,
        "interruptionNote":  f"Point de départ réinitialisé par {ic} interruption(s)." if ic>0 else None,
    }

def normalize_dataframe_rows(value):
    if value is None: return []
    if hasattr(value,"values"): return value.values.tolist()
    return value if isinstance(value,list) else []

def claim_choices_for_category(category):
    claims = CIVIL_CLAIMS if category=="civil" else PENAL_CLAIMS
    return [(f"{CLAIM_LABELS[c]['fr']} / {CLAIM_LABELS[c]['ar']}", c) for c in claims]

def on_claim_category_change(category):
    choices = claim_choices_for_category(category)
    return gr.update(choices=choices, value=choices[0][1])

def on_adjustments_toggle(enabled):
    return gr.update(visible=bool(enabled)), gr.update(visible=bool(enabled))

def prescription_calculate(claim_category, claim_type, start_date_text, adjustments_enabled, suspension_rows, interruption_rows):
    valid = CIVIL_CLAIMS if claim_category=="civil" else PENAL_CLAIMS
    if claim_type not in valid: claim_type = valid[0]
    start_date = parse_date(start_date_text)
    if not start_date:
        return "<div class='soft-card'>Date de départ invalide.</div>","","",""

    suspensions, interruptions = [], []
    if adjustments_enabled:
        for row in normalize_dataframe_rows(suspension_rows):
            row = row if isinstance(row,list) else []
            stype = str(row[0]).strip() if len(row)>0 and row[0] is not None else "force_majeure"
            sd, ed = parse_date(row[1] if len(row)>1 else None), parse_date(row[2] if len(row)>2 else None)
            if sd and ed: suspensions.append({"type":stype,"startDate":sd,"endDate":ed})
        for row in normalize_dataframe_rows(interruption_rows):
            row = row if isinstance(row,list) else []
            itype = str(row[0]).strip() if len(row)>0 and row[0] is not None else "legal_citation"
            id_ = parse_date(row[1] if len(row)>1 else None)
            if id_: interruptions.append({"type":itype,"date":id_})

    result = calculate_prescription({"claimType":claim_type,"startDate":start_date,"suspensions":suspensions if adjustments_enabled else [],"interruptions":interruptions if adjustments_enabled else []})
    cl = CLAIM_LABELS[result["claimType"]]
    result_html = (
        "<div class='metric-card'>"
        f"<div class='calc-title'>{html.escape(cl['fr'])} / {html.escape(cl['ar'])}</div>"
        f"<div><b>Référence :</b> {html.escape(result['articleRef'])}</div>"
        f"<div><b>Délai légal :</b> {result['prescriptionYears']} an(s)</div>"
        f"<div><b>Date d'expiration :</b> {fmt_date(result['expiryDate'])}</div>"
        f"<div><b>Jours restants :</b> {result['daysRemaining']}</div>"
        f"<div style='margin-top:8px'>{status_badge_html(result['status'])}</div>"
        "</div>"
    )
    timeline_html = (
        "<div class='soft-card'><div class='calc-title'>Timeline</div>"
        f"<div>{fmt_date(result['adjustedStartDate'])} → {fmt_date(result['expiryDate'])}</div></div>"
    )
    notes = [n for n in [result.get("suspensionNote"),result.get("interruptionNote")] if n]
    notes_html = "<div class='soft-card'>"+"<br>".join(html.escape(n) for n in notes)+"</div>" if notes else ""
    summary = (
        f"Explique ce calcul de prescription en droit tunisien pour '{cl['fr']}': "
        f"début {fmt_date(result['adjustedStartDate'])}, expiration {fmt_date(result['expiryDate'])}, "
        f"statut {result['status']}."
    )
    return result_html, timeline_html, notes_html, summary

def prescription_reset():
    return ["civil","civil_personal_general",today_string(),False,[],[],  "","","","",""]


# ── Appeal deadlines logic ────────────────────────────────────────────────────
def apply_start_rule(base_date, notification_method):
    return add_days(base_date,1) if notification_method=="direct_party" else base_date

def notification_modifier_days(nm):
    return 10 if nm=="parquet" else (3 if nm=="greffe" else 0)

def status_for_deadline(deadline):
    rem = days_between(date.today(), deadline)
    if rem<0:  return {"status":"expired","daysRemaining":rem}
    if rem<=15:return {"status":"warning","daysRemaining":rem}
    return {"status":"active","daysRemaining":rem}

def build_available_remedy(template, start_date, notification_method, extra_days=0):
    modifier  = notification_modifier_days(notification_method)+(extra_days or 0)
    base_days = template["days"] if template["days"] is not None else (template["months"] or 0)*30
    day_delta = base_days + modifier
    raw = add_days(add_months(start_date,template["months"]),modifier) if template["months"] is not None else add_days(start_date,day_delta)
    adjusted  = adjust_to_next_working_day(raw)
    si        = status_for_deadline(adjusted)
    return {"name":template["name"],"nameAr":template["nameAr"],"available":True,"deadline":adjusted,"deadlineDays":day_delta,"articleRef":template["articleRef"],"status":si["status"],"daysRemaining":si["daysRemaining"]}

def build_unavailable(name, name_ar, reason):
    return {"name":name,"nameAr":name_ar,"available":False,"deadline":None,"deadlineDays":0,"articleRef":"Vérification manuelle","unavailableReason":reason}

def calculate_appeal_deadlines(input_data):
    jd = input_data["judgmentDate"]
    nd = input_data.get("notificationDate") or jd
    remedies = []
    cl = input_data["courtLevel"]; matter = input_data["matter"]

    if cl=="first_instance":
        if matter=="penal":       remedies.append(build_available_remedy(APPEL_PENAL_TEMPLATE, jd, input_data["notificationMethod"]))
        elif matter=="administratif": remedies.append(build_available_remedy(APPEL_ADMIN_TEMPLATE, apply_start_rule(nd,input_data["notificationMethod"]), input_data["notificationMethod"]))
        elif matter in ("civil","commercial"):
            start = apply_start_rule(nd, input_data["notificationMethod"])
            if input_data["judgmentNature"]=="par_defaut":
                remedies.append(build_available_remedy(OPPOSITION_TEMPLATE, start, input_data["notificationMethod"]))
                remedies.append(build_available_remedy(APPEL_CIVIL_TEMPLATE, start, input_data["notificationMethod"], extra_days=15))
            else:
                remedies.append(build_available_remedy(APPEL_CIVIL_TEMPLATE, start, input_data["notificationMethod"]))
    elif cl=="cour_appel":
        if matter in ("civil","commercial"):
            remedies.append(build_available_remedy(CASSATION_CIVIL_TEMPLATE, apply_start_rule(nd,input_data["notificationMethod"]), input_data["notificationMethod"]))
        elif matter=="penal":
            remedies.append(build_available_remedy(CASSATION_PENAL_TEMPLATE, jd, input_data["notificationMethod"]))
        else:
            remedies.append(build_unavailable("Pourvoi en cassation","الطعن بالتعقيب","Matière non prévue pour la cassation."))
    elif cl=="cour_cassation":
        remedies.append(build_unavailable("Recours ordinaire","طعن عادي","Aucune voie de recours ordinaire supplémentaire."))

    if not remedies:
        remedies.append(build_unavailable("Aucun recours standard","لا يوجد طعن معياري","Vérifiez la combinaison juridiction/matière/nature du jugement."))

    timeline_points = [
        {"date":jd,"label":"Date du jugement","type":"start"},
        {"date":nd,"label":"Date de notification","type":"info"},
    ]
    for r in remedies:
        if r.get("available") and r.get("deadline"):
            timeline_points.append({"date":r["deadline"],"label":f"{r['name']} ({r['deadlineDays']} j)","type":"deadline"})
    timeline_points.sort(key=lambda x: x["date"])

    future = [r for r in remedies if r.get("available") and isinstance(r.get("daysRemaining"),int) and r["daysRemaining"]>=0]
    future.sort(key=lambda x: x["daysRemaining"])
    return {"remedies":remedies,"timelinePoints":timeline_points,"nextUrgentDeadline":future[0] if future else None}

def appeal_calculate(court_level, matter, judgment_nature, judgment_date_text, notification_method, notification_date_text):
    jd = parse_date(judgment_date_text)
    if not jd: return "<div class='soft-card'>Date du jugement invalide.</div>","","",""
    nd = parse_date(notification_date_text) or jd
    result = calculate_appeal_deadlines({"courtLevel":court_level,"matter":matter,"judgmentNature":judgment_nature,"judgmentDate":jd,"notificationMethod":notification_method,"notificationDate":nd})

    cards_html = ""
    for remedy in result["remedies"]:
        status_h  = status_badge_html(remedy.get("status")) if remedy.get("available") else ""
        urgency_h = urgent_badge_html(remedy.get("daysRemaining"))
        unavail   = f"<div class='soft-card' style='margin-top:8px'>{html.escape(remedy.get('unavailableReason',''))}</div>" if not remedy.get("available") else ""
        cards_html += (
            "<div class='metric-card' style='margin-bottom:8px'>"
            f"<div class='calc-title'>{html.escape(remedy['name'])} / {html.escape(remedy['nameAr'])}</div>"
            f"<div style='display:flex;gap:8px;align-items:center;margin-bottom:6px'>{status_h}{urgency_h}</div>"
            f"<div><b>Délai :</b> {remedy['deadlineDays']} jour(s)</div>"
            f"<div><b>Échéance :</b> {fmt_date(remedy.get('deadline'))}</div>"
            f"<div><b>Restant :</b> {remedy.get('daysRemaining','-')} jour(s)</div>"
            f"<div style='font-size:12px;margin-top:4px'>{html.escape(remedy['articleRef'])}</div>"
            f"{unavail}</div>"
        )

    timeline_html = "<div class='soft-card'><div class='calc-title'>Timeline</div><ul>"
    for pt in result["timelinePoints"]:
        timeline_html += f"<li>{fmt_date(pt['date'])} – {html.escape(pt['label'])}</li>"
    timeline_html += "</ul></div>"

    urgent_html = ""
    if result.get("nextUrgentDeadline"):
        nxt = result["nextUrgentDeadline"]
        urgent_html = f"<div class='soft-card'>Prochain délai urgent : {html.escape(nxt['name'])} ({nxt.get('daysRemaining','?')} jour(s)).</div>"

    summary_items = [f"{r['name']}: {r.get('daysRemaining','?')} jours" for r in result["remedies"] if r.get("available")]
    summary = "Explique ces délais de recours en procédure tunisienne : " + "; ".join(summary_items)
    return cards_html, timeline_html, urgent_html, summary

def appeal_reset():
    return ["first_instance","civil","contradictoire",today_string(),"direct_party",today_string(),"","","","",""]


# ── Build UI ──────────────────────────────────────────────────────────────────
initial_state          = init_chat_state()
initial_sidebar_choices= build_sidebar_choices(initial_state)

custom_theme = gr.themes.Base(primary_hue="amber", secondary_hue="orange", neutral_hue="stone")

with gr.Blocks(theme=custom_theme, css=CSS, head=HEAD_JS, title="LEGAL MIND") as demo:
    app_state = gr.State(initial_state)

    with gr.Tabs():

        # ── Chat tab ──────────────────────────────────────────────────────────
        with gr.Tab("💬 Chat", id="tab_chat"):
            gr.HTML(
                f"""
                <div id='chat-header'>
                  <div class='brand-wrap'>
                    {LOGO_HTML}
                    <div class='brand-title'>LEGAL MIND</div>
                  </div>
                  <div style='font-size:12px;color:#E7C98E'>
                    Calculateurs ⟶ onglets &nbsp;⚖️ &nbsp;⏱️ &nbsp;📅
                  </div>
                </div>
                """
            )

            with gr.Row():
                # Sidebar
                with gr.Column(scale=1, min_width=260):
                    gr.HTML("<div class='chat-side-card'><b>Historique / السجل</b></div>")
                    new_conv_btn   = gr.Button("New conversation", variant="primary")
                    clear_hist_btn = gr.Button("Clear history",    variant="stop")
                    conversation_radio = gr.Radio(
                        choices=initial_sidebar_choices,
                        value=initial_state["active_id"],
                        label="Conversations",
                    )

                # Main chat column
                with gr.Column(scale=3):
                    # Custom HTML chat display (avoids all gr.Chatbot version issues)
                    chat_messages = gr.HTML(
                        value=render_chat_html(initial_state),
                        elem_id="chat_messages",
                    )

                    with gr.Group(visible=True) as examples_group:
                        gr.HTML("<div class='empty-shell'><b>Exemples / أمثلة</b></div>")
                        with gr.Row():
                            with gr.Column():
                                gr.Markdown("**Arabic**")
                                ex_ar_btns = [gr.Button(t, size="sm") for t in AR_EXAMPLES]
                            with gr.Column():
                                gr.Markdown("**Français**")
                                ex_fr_btns = [gr.Button(t, size="sm") for t in FR_EXAMPLES]

                    with gr.Row(equal_height=True):
                        input_lang_badge = gr.HTML(render_input_lang_badge(""))
                        chat_input = gr.Textbox(
                            value="", lines=2, max_lines=6,
                            label="", placeholder=PLACEHOLDERS[0],
                            elem_id="chat_input", show_label=False, scale=7,
                        )
                        send_btn = gr.Button("Send", variant="primary", elem_id="send_btn", scale=1)

                    char_counter = gr.HTML(render_counter(""))

                    with gr.Accordion("Sources (0)", open=False) as sources_acc:
                        sources_html_comp = gr.HTML(render_sources_panel([]))

        # ── Mirath tab ────────────────────────────────────────────────────────
        with gr.Tab("⚖️ Mirath", id="tab_mirath"):
            gr.Markdown("### Mirath Calculator / Calcul des parts héréditaires")
            with gr.Row():
                mirath_gender = gr.Dropdown(
                    choices=[("Male / ذكر","male"),("Female / أنثى","female")],
                    value="male", label="Deceased gender",
                )
                mirath_estate = gr.Number(label="Estate value (TND, optional)", precision=3)

            mirath_inputs = {}
            for idx in range(0, len(HEIR_ORDER), 3):
                with gr.Row():
                    for htype in HEIR_ORDER[idx:idx+3]:
                        labels = HEIR_LABELS[htype]
                        mirath_inputs[htype] = gr.Number(
                            label=f"{labels['fr']} / {labels['ar']}",
                            value=0, precision=0, minimum=0, maximum=20,
                        )

            with gr.Row():
                mirath_calc_btn  = gr.Button("Calculer",     variant="primary")
                mirath_reset_btn = gr.Button("Réinitialiser")

            mirath_table = gr.Dataframe(
                headers=["Héritier / الوارث","Part","%","Montant (TND)"],
                datatype=["str","str","str","str"], value=[], interactive=False, wrap=True,
            )
            mirath_badges        = gr.HTML()
            mirath_notes         = gr.HTML()
            mirath_summary_state = gr.State("")
            with gr.Row():
                mirath_explain_btn = gr.Button("Explain with AI", variant="secondary")
                mirath_explain_out = gr.Textbox(label="AI explanation", lines=6)

        # ── Prescription tab ──────────────────────────────────────────────────
        with gr.Tab("⏱️ Prescription", id="tab_prescription"):
            gr.Markdown("### Calculateur de prescription / حاسبة التقادم")
            with gr.Row():
                claim_category = gr.Dropdown(
                    choices=[("Civil / مدني","civil"),("Pénal / جزائي","penal")],
                    value="civil", label="Catégorie",
                )
                claim_type = gr.Dropdown(
                    choices=claim_choices_for_category("civil"),
                    value="civil_personal_general", label="Type d'action",
                )
            with gr.Row():
                prescription_start_date = gr.Textbox(value=today_string(), label="Date de départ (AAAA-MM-JJ)")
                adjustments_enabled     = gr.Checkbox(value=False, label="Activer suspensions / interruptions")
            with gr.Column(visible=False) as susp_wrap:
                suspension_df = gr.Dataframe(
                    headers=["type","start_date","end_date"], datatype=["str","str","str"],
                    row_count=(0,"dynamic"), col_count=(3,"fixed"), value=[], interactive=True,
                    label="Événements de suspension",
                )
            with gr.Column(visible=False) as intr_wrap:
                interruption_df = gr.Dataframe(
                    headers=["type","date"], datatype=["str","str"],
                    row_count=(0,"dynamic"), col_count=(2,"fixed"), value=[], interactive=True,
                    label="Événements d'interruption",
                )
            with gr.Row():
                prescription_calc_btn  = gr.Button("Calculer",     variant="primary")
                prescription_reset_btn = gr.Button("Réinitialiser")

            prescription_result_html   = gr.HTML()
            prescription_timeline_html = gr.HTML()
            prescription_notes_html    = gr.HTML()
            prescription_summary_state = gr.State("")
            with gr.Row():
                prescription_explain_btn = gr.Button("Explain with AI", variant="secondary")
                prescription_explain_out = gr.Textbox(label="AI explanation", lines=6)

        # ── Recours tab ───────────────────────────────────────────────────────
        with gr.Tab("📅 Recours", id="tab_recours"):
            gr.Markdown("### Délais de recours / آجال الطعون")
            with gr.Row():
                court_level = gr.Dropdown(
                    choices=[(f"{v['fr']} / {v['ar']}",k) for k,v in COURT_LABELS.items()],
                    value="first_instance", label="Niveau de juridiction",
                )
                matter = gr.Dropdown(
                    choices=[(f"{v['fr']} / {v['ar']}",k) for k,v in MATTER_LABELS.items()],
                    value="civil", label="Matière",
                )
            with gr.Row():
                judgment_nature = gr.Dropdown(
                    choices=[(f"{v['fr']} / {v['ar']}",k) for k,v in JUDGMENT_NATURE_LABELS.items()],
                    value="contradictoire", label="Nature du jugement",
                )
                judgment_date = gr.Textbox(value=today_string(), label="Date du jugement (AAAA-MM-JJ)")
            with gr.Row():
                notification_method = gr.Dropdown(
                    choices=[(f"{v['fr']} / {v['ar']}",k) for k,v in NOTIFICATION_LABELS.items()],
                    value="direct_party", label="Mode de notification",
                )
                notification_date = gr.Textbox(value=today_string(), label="Date de notification (AAAA-MM-JJ)")
            with gr.Row():
                appeal_calc_btn  = gr.Button("Calculer",     variant="primary")
                appeal_reset_btn = gr.Button("Réinitialiser")

            appeal_cards_html    = gr.HTML()
            appeal_timeline_html = gr.HTML()
            appeal_urgent_html   = gr.HTML()
            appeal_summary_state = gr.State("")
            with gr.Row():
                appeal_explain_btn = gr.Button("Explain with AI", variant="secondary")
                appeal_explain_out = gr.Textbox(label="AI explanation", lines=6)


    # ── Event wiring ──────────────────────────────────────────────────────────
    # chat_messages is now gr.HTML — its output is a plain HTML string
    chat_core_outputs = [chat_messages, app_state, conversation_radio, examples_group, sources_acc, sources_html_comp]
    chat_send_outputs = [chat_messages, app_state, conversation_radio, examples_group, sources_acc, sources_html_comp, chat_input, send_btn]

    conversation_radio.change(fn=on_select_conversation, inputs=[conversation_radio, app_state], outputs=chat_core_outputs, queue=False)
    new_conv_btn.click(fn=on_new_conversation, inputs=[app_state], outputs=chat_core_outputs, queue=False)
    clear_hist_btn.click(fn=on_clear_history, inputs=[app_state], outputs=chat_core_outputs, queue=False)

    chat_input.input(fn=on_input_change, inputs=[chat_input], outputs=[chat_input, input_lang_badge, char_counter], queue=False)
    send_btn.click(fn=on_send_message, inputs=[chat_input, app_state], outputs=chat_send_outputs)
    chat_input.submit(fn=on_send_message, inputs=[chat_input, app_state], outputs=chat_send_outputs)

    for text, btn in zip(AR_EXAMPLES + FR_EXAMPLES, ex_ar_btns + ex_fr_btns):
        btn.click(fn=lambda t=text: t, inputs=None, outputs=[chat_input], queue=False).then(
            fn=on_send_message, inputs=[chat_input, app_state], outputs=chat_send_outputs,
        )

    # Mirath
    mirath_calc_btn.click(
        fn=mirath_calculate,
        inputs=[mirath_gender, mirath_estate] + [mirath_inputs[h] for h in HEIR_ORDER],
        outputs=[mirath_table, mirath_badges, mirath_notes, mirath_summary_state],
    )
    mirath_reset_btn.click(
        fn=mirath_reset, inputs=None,
        outputs=[mirath_gender, mirath_estate, *[mirath_inputs[h] for h in HEIR_ORDER],
                 mirath_table, mirath_badges, mirath_notes, mirath_summary_state],
        queue=False,
    )
    mirath_explain_btn.click(
        fn=push_summary_to_chat, inputs=[mirath_summary_state, app_state],
        outputs=[mirath_explain_out, *chat_core_outputs],
    )

    # Prescription
    claim_category.change(fn=on_claim_category_change, inputs=[claim_category], outputs=[claim_type], queue=False)
    adjustments_enabled.change(fn=on_adjustments_toggle, inputs=[adjustments_enabled], outputs=[susp_wrap, intr_wrap], queue=False)
    prescription_calc_btn.click(
        fn=prescription_calculate,
        inputs=[claim_category, claim_type, prescription_start_date, adjustments_enabled, suspension_df, interruption_df],
        outputs=[prescription_result_html, prescription_timeline_html, prescription_notes_html, prescription_summary_state],
    )
    prescription_reset_btn.click(
        fn=prescription_reset, inputs=None,
        outputs=[claim_category, claim_type, prescription_start_date, adjustments_enabled,
                 suspension_df, interruption_df, prescription_result_html, prescription_timeline_html,
                 prescription_notes_html, prescription_summary_state, prescription_explain_out],
        queue=False,
    )
    prescription_explain_btn.click(
        fn=push_summary_to_chat, inputs=[prescription_summary_state, app_state],
        outputs=[prescription_explain_out, *chat_core_outputs],
    )

    # Recours
    appeal_calc_btn.click(
        fn=appeal_calculate,
        inputs=[court_level, matter, judgment_nature, judgment_date, notification_method, notification_date],
        outputs=[appeal_cards_html, appeal_timeline_html, appeal_urgent_html, appeal_summary_state],
    )
    appeal_reset_btn.click(
        fn=appeal_reset, inputs=None,
        outputs=[court_level, matter, judgment_nature, judgment_date, notification_method, notification_date,
                 appeal_cards_html, appeal_timeline_html, appeal_urgent_html, appeal_summary_state, appeal_explain_out],
        queue=False,
    )
    appeal_explain_btn.click(
        fn=push_summary_to_chat, inputs=[appeal_summary_state, app_state],
        outputs=[appeal_explain_out, *chat_core_outputs],
    )

    demo.load(fn=lambda s: compose_chat_outputs(s), inputs=[app_state], outputs=chat_core_outputs, queue=False)


# ── Launch ────────────────────────────────────────────────────────────────────
demo.queue().launch(share=True, debug=True)


Gradio 5.50.0  (major = 5)


/tmp/ipykernel_13284/2710048723.py:1434: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=custom_theme, css=CSS, head=HEAD_JS, title="LEGAL MIND") as demo:
/tmp/ipykernel_13284/2710048723.py:1434: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=custom_theme, css=CSS, head=HEAD_JS, title="LEGAL MIND") as demo:
/tmp/ipykernel_13284/2710048723.py:1434: DeprecationWarning: The 'head' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'head' to Blocks.launch() instead.
  with gr.Blocks(theme=custom_theme, css=CSS, head=HEAD_JS, title="LEGAL MIND") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9e0b96766fc38dbf8c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=3

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9e0b96766fc38dbf8c.gradio.live
